# Wildlife Re-Identification — Standalone Kaggle Pipeline (`v25_hnsw_rescue` + `v26_dual_r2` Blend)

**Latest Dual-Paradigm Architecture (`src/reid/`):**
1. **Open-Set Encounter-Aware CV:** 5 primary splits (`enc_thr=0.2`, `p_move={hyena:0.30, leopard:0.25}`, `f_new_id={hyena:0.15, leopard:0.40}`) + optional mixed-difficulty `extra_cv` training splits (`split + 10`) strictly excluded from calibration, evaluation, and tuning.
2. **HNSW `99.1%`-Recall LNBNN (`lnbnn_hnsw`):** Vectorized `k=50` RootSIFT neighbor cache yielding `lnb` ($K=10$), `lnb_k3` ($K=3$), `lnb_img` (image-max), `lnb_uimg` (cross-identity background-suppressed), and `lnb_rimg` (strict `0.85` cross-ID ratio).
3. **Gallery-Only Metric Learning & Stage-1 Rescue:** Per-fold gallery-only `sw2152` (full 2152-dim $S_w^{-1/2}$ whitening), `ssw2152` (side-conditioned), `fl512` (full-dim Fisher LDA), `cl2`, `pca512->lda256`, `krec_sw10`, `diff_sw`, gallery-only `r_g_diff` hubness, and viewpoint compatibility `z`-scores.
4. **Two-Stage Camera-Trap SIFT Background Filtering:** RootSIFT + mutual Lowe ratio (`0.8`) + OpenCV RANSAC (`estimateAffinePartial2D` + `findHomography`) with Stage-1 corner killer (`abs_rot < 0.50° & spread_min < 0.25`) and Stage-2 full-frame killer (`abs_rot < 0.50° & lnb_uimg_share < 0.35 & inl >= 6`).
5. **Paradigm 1 (`v25_hnsw_rescue`):** Single-round monotone `LGBModel` + per-species `P(new)` ensemble + HNSW SIFT keypoint query graph + `sw2152`/`fl512` query affinity + nested-tuned Zhou label propagation (`conn_new_scale`).
6. **Paradigm 2 (`v26_dual_r2`):** Two-round neighbor-stacked 4-family ensemble (`UnconstrainedLGB` + `ExtraTreesLGB` + `LambdaRankLGB` + `SGDLogModel`, plus optional GPU `XGBoost`) -> fold-isolated Round-2 query-neighbor stacking (`k_nb=8`) -> Round-2 ensemble + per-species `P(new)` + nested-tuned transductive propagation.
7. **Honest Nested CV Blend:** Evaluates leave-one-split-out blend weights and generates validated `submission.csv` + `.probs.pkl` files in `/kaggle/working`.

### Compliance & Speed Notes
- **100% Competition-Compliant:** Uses ONLY allowlisted libraries (`numpy`, `pandas`, `scipy`, `scikit-learn`, `lightgbm`, `xgboost`, `networkx`, `opencv-python` matching/RANSAC only, `faiss-cpu`). Zero `torch`, `cupy`, `faiss-gpu`, or image files are used.
- **Fastest Kaggle Run (Optional Cache Dataset):** If you upload your precomputed `sift_pairs_r0.8_f0.02.npz` (and optionally `lnbnn_hnsw/` or `sift_xy/desc/off.npy`) as a private Kaggle Dataset and attach it to the notebook, **Cell 1 auto-detects it in `/kaggle/input/`** and skips hours of SIFT RANSAC recomputation (full `v25+v26` finishes in **~10–15 minutes**!).
- **Running From Scratch on Kaggle:** Set `MODE = "kaggle_fast"` (2 extra CV splits, 5 imgs/id for SIFT) if running from scratch without a precomputed SIFT cache, or `MODE = "full"` for the exact `v25+v26` configuration.

In [ ]:
import ast
import glob
import hashlib
import json
import multiprocessing as mp
import os
import pickle
import shutil
import subprocess
import sys
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import lightgbm as lgb
import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import squareform
from scipy.special import expit, logit
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

# Ensure faiss-cpu is available on Kaggle (falls back to pure-NumPy if offline & uninstalled)
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss
        HAS_FAISS = True
    except Exception as _e:
        print(f"[warn] faiss-cpu unavailable ({_e}); will use chunked NumPy L2 fallback for LNBNN/kpgraph.")

SPECIES = ("hyena", "leopard")
NEW = "new_individual"
W_TEST = {"hyena": 1217 / 2791, "leopard": 1574 / 2791}
N_THREADS = os.cpu_count() or 4
if HAS_FAISS:
    faiss.omp_set_num_threads(N_THREADS)


def has_cuda_gpu():
    try:
        out = subprocess.check_output(["nvidia-smi", "-L"], stderr=subprocess.DEVNULL, text=True)
        return "GPU" in out
    except Exception:
        return False


HAS_CUDA = has_cuda_gpu()

# ==============================================================================
# USER CONFIGURATION
# ==============================================================================
MODE = "full"             # "full" = exact v25+v26 config; "kaggle_fast" = lighter SIFT/extra_cv when running without cache
USE_GPU_XGB = False       # True (when HAS_CUDA) adds GPU-accelerated XGBoost (device="cuda") to Paradigm 2 ensemble
BLEND_MODE = "auto"       # "auto" (nested CV optimal per-species weight, or 0.5/0.5 if tie), "equal" (0.5/0.5), or float weight for v25
FORCE_RERUN = False       # True ignores cached v25/v26 .probs.pkl and retrains models from tables
SIFT_WORKERS = N_THREADS  # Parallel workers for missing SIFT pairs

# ==============================================================================
# AUTO-DISCOVER DATA & PRECOMPUTED CACHES (Kaggle /kaggle/input or local data/)
# ==============================================================================
SEARCH_ROOTS = [Path("/kaggle/input"), Path("data"), Path("."), Path("..")]
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
CACHE_DIR = Path("data/cache") if (WORK_DIR == Path(".") and Path("data/cache").is_dir()) else (WORK_DIR / "cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _find_first(pattern, is_dir=False):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for hit in sorted(root.rglob(pattern)):
            if is_dir and hit.is_dir():
                return hit
            if not is_dir and hit.is_file():
                return hit
    return None


def _find_sift_source(kind):
    # Matches either sift_features_train.npz file or a directory containing tr_*/te_* .npy files
    fname = f"sift_features_{kind}.npz"
    hit = _find_first(fname, is_dir=False)
    if hit is not None:
        return hit
    hit_dir = _find_first(fname, is_dir=True)
    if hit_dir is not None:
        return hit_dir
    prefix = "tr_*_xy.npy" if kind == "train" else "te_*_xy.npy"
    sample = _find_first(prefix, is_dir=False)
    if sample is not None:
        return sample.parent
    raise FileNotFoundError(f"Could not locate {fname} in {SEARCH_ROOTS}")


RAW_DIR = _find_first("miewid_features.npy", is_dir=False)
if RAW_DIR is None:
    raise FileNotFoundError(f"Could not find miewid_features.npy in {SEARCH_ROOTS}")
RAW_DIR = RAW_DIR.parent

SIFT_TRAIN_PATH = _find_sift_source("train")
SIFT_TEST_PATH = _find_sift_source("test")

# Optional precomputed caches attached via Kaggle Dataset or local data/cache/
PRE_SIFT_OFF = _find_first("sift_off.npy", is_dir=False)
PRE_SIFT_PACK_DIR = PRE_SIFT_OFF.parent if (PRE_SIFT_OFF and (PRE_SIFT_OFF.parent / "sift_xy.npy").exists() and (PRE_SIFT_OFF.parent / "sift_desc.npy").exists()) else None
PRE_SIFT_PAIRS = _find_first("sift_pairs_r0.8_f0.02.npz", is_dir=False)
PRE_LNBNN_HNSW = _find_first("lnbnn_hnsw", is_dir=True)
if PRE_LNBNN_HNSW and (PRE_LNBNN_HNSW / "hyena_meta.json").exists() and (PRE_LNBNN_HNSW / "leopard_meta.json").exists():
    PRE_LNBNN_DIR = PRE_LNBNN_HNSW
else:
    PRE_LNBNN_META = _find_first("leopard_meta.json", is_dir=False)
    PRE_LNBNN_DIR = PRE_LNBNN_META.parent if (PRE_LNBNN_META and (PRE_LNBNN_META.parent / "hyena_meta.json").exists()) else None
PRE_KPGRAPH_DIR = _find_first("kpgraph", is_dir=True)
PRE_V25_PROBS = _find_first("v25.probs.pkl", is_dir=False)
PRE_V25_CV_PROBS = _find_first("v25.cv.probs.pkl", is_dir=False)
PRE_V26_PROBS = _find_first("v26.probs.pkl", is_dir=False)
PRE_V26_CV_PROBS = _find_first("v26.cv.probs.pkl", is_dir=False)


def build_configs(mode="full"):
    base_cv = {
        "seeds": [0, 1, 2, 3, 4],
        "enc_thr": 0.2,
        "p_move": {"hyena": 0.3, "leopard": 0.25},
        "f_new_id": {"hyena": 0.15, "leopard": 0.4},
        "max_new_id_size": 40,
    }
    extra_cv = {
        "seeds": [0, 1, 2, 3, 4] if mode == "full" else [0, 1],
        "enc_thr": {"hyena": 0.2, "leopard": 0.3},
        "p_move": {"hyena": 0.5, "leopard": 0.5},
        "f_new_id": {"hyena": 0.1, "leopard": 0.2},
        "max_new_id_size": 1000000,
    }
    retrieval = {
        "embedding": "l2",
        "n_cand": 30,
        "tau": {"hyena": 0.49, "leopard": 0.52},
        "lnbnn": True,
        "lnbnn_img": True,
        "lnbnn_cand": 8,
        "lnbnn_ucand": 12,
        "lnbnn_sift_imgs": 2,
        "lnbnn_uimg_sift_imgs": 1,
        "lnbnn_nprobe": 64,
        "krec": True,
        "bg_clean": True,
        "v25_stage1": True,
        "bg_clean_v2_img": True,
        "sw_cand": 15 if mode == "full" else 10,
        "fl_cand": 15 if mode == "full" else 10,
        "krec_sw_cand": 10 if mode == "full" else 8,
        "sift_imgs_sw": 2,
    }
    sift_cfg = {
        "enabled": True,
        "ratio": 0.8,
        "ransac_frac": 0.02,
        "top_ids": 15,
        "imgs_per_id": 8 if mode == "full" else 5,
        "graph_k": 10,
        "workers": SIFT_WORKERS,
    }
    trans_cfg = {
        "enabled": True,
        "features": True,
        "consensus": False,
        "alphas": [0.0, 0.4, 0.6, 0.75, 0.85],
        "tau_e": [0.35, 0.45, 0.55, 0.65, 0.75],
        "kp_graph": True,
        "kp_hnsw": True,
        "kp_nprobe": 64,
        "trans_sw": True,
        "sift_bg": True,
        "sift_bg_v2": True,
        "conn_new_scale": True,
        "nested_tune": True,
    }
    lnb_dir = str(PRE_LNBNN_DIR) if PRE_LNBNN_DIR is not None else str(CACHE_DIR / "lnbnn_hnsw")
    cfg_v25 = {
        "exp_slug": "v25_hnsw_rescue",
        "seed": 2026,
        "lnbnn_dir": lnb_dir,
        "cv": base_cv,
        "extra_cv": extra_cv,
        "retrieval": retrieval,
        "sift": sift_cfg,
        "model": {
            "enabled": True,
            "methods": ["softmax"],
            "final_method": "softmax",
            "bg_clean_v2": True,
            "hubness": True,
            "lnb_rimg": True,
            "rich_q": True,
            "pnew_per_species": True,
        },
        "transductive": trans_cfg,
    }
    cfg_v26 = {
        "exp_slug": "v26_dual_r2",
        "seed": 2026,
        "lnbnn_dir": lnb_dir,
        "cv": base_cv,
        "extra_cv": extra_cv,
        "retrieval": retrieval,
        "sift": sift_cfg,
        "model": {
            "enabled": True,
            "paradigm": "dual",
            "methods": ["softmax"],
            "final_method": "softmax",
            "bg_clean_v2": True,
            "hubness": True,
            "lnb_rimg": True,
            "rich_q": True,
            "r2_k": 8,
            "bag_seeds": [2026],
            "use_gpu_xgb": bool(USE_GPU_XGB and HAS_CUDA),
        },
        "transductive": trans_cfg,
    }
    return cfg_v25, cfg_v26


CFG_V25, CFG_V26 = build_configs(MODE)
T_START = time.time()
print(f"Mode={MODE} | CPUs={N_THREADS} | FAISS={HAS_FAISS} | CUDA_GPU={HAS_CUDA} (USE_GPU_XGB={CFG_V26['model']['use_gpu_xgb']})")
print(f"RAW_DIR       : {RAW_DIR}")
print(f"SIFT_TRAIN    : {SIFT_TRAIN_PATH}")
print(f"SIFT_TEST     : {SIFT_TEST_PATH}")
print(f"WORK_DIR      : {WORK_DIR}")
print(f"PRE_SIFT_PACK : {PRE_SIFT_PACK_DIR}")
print(f"PRE_SIFT_PAIRS: {PRE_SIFT_PAIRS}")
print(f"LNBNN_DIR     : {CFG_V25['lnbnn_dir']} (precomputed={PRE_LNBNN_DIR is not None})")

In [ ]:
@dataclass
class Data:
    meta: pd.DataFrame
    X: np.ndarray
    ntr: int

    @property
    def train(self):
        return self.meta.iloc[: self.ntr]

    @property
    def test(self):
        return self.meta.iloc[self.ntr:]


def load_data() -> Data:
    tr = pd.read_csv(RAW_DIR / "train.csv", dtype=str)
    te = pd.read_csv(RAW_DIR / "test.csv", dtype=str)
    ids = pd.read_csv(RAW_DIR / "miewid_ids.csv", dtype=str)["id"]
    X = np.load(RAW_DIR / "miewid_features.npy")
    assert X.shape == (len(ids), 2152) and X.dtype == np.float32
    assert not tr.id.duplicated().any() and not te.id.duplicated().any() and not ids.duplicated().any()
    assert set(ids) == set(tr.id) | set(te.id) and len(ids) == len(tr) + len(te)
    assert np.isfinite(X).all()
    meta = pd.concat([tr.assign(split="train"), te.assign(split="test", identity=np.nan)], ignore_index=True)
    pos = pd.Series(np.arange(len(ids)), index=ids.values)
    X = X[pos.loc[meta.id].values]
    meta["row"] = np.arange(len(meta))
    for sp in meta.species.unique():
        assert sp in SPECIES, sp
    assert (meta[meta.split == "train"].groupby("identity").species.nunique() == 1).all()
    return Data(meta=meta, X=X, ntr=len(tr))


def _sift_pack_paths():
    d = PRE_SIFT_PACK_DIR if PRE_SIFT_PACK_DIR is not None else CACHE_DIR
    return d / "sift_xy.npy", d / "sift_desc.npy", d / "sift_off.npy"


def build_sift_pack(data: Data):
    pxy, pde, poff = _sift_pack_paths()
    if poff.exists() and pxy.exists() and pde.exists():
        print(f"[sift_pack] using cached {pxy.parent}")
        return
    t0 = time.time()
    pxy, pde, poff = CACHE_DIR / "sift_xy.npy", CACHE_DIR / "sift_desc.npy", CACHE_DIR / "sift_off.npy"
    xs, ds, off = [], [], [0]
    src = {}
    for key, p in (("train", SIFT_TRAIN_PATH), ("test", SIFT_TEST_PATH)):
        src[key] = np.load(p) if p.is_file() else p

    def _get(key, sid, kind):
        s = src[key]
        return s[f"{sid}_{kind}"] if not isinstance(s, Path) else np.load(s / f"{sid}_{kind}.npy")

    for sid in data.meta.id:
        key = "train" if sid.startswith("tr_") else "test"
        xy = _get(key, sid, "xy")
        de = _get(key, sid, "desc")
        if len(xy) == 0:
            xy = np.zeros((0, 2), np.float32)
            de = np.zeros((0, 128), np.uint8)
        assert xy.dtype == np.float32 and xy.shape[1] == 2 and de.dtype == np.uint8 and de.shape == (len(xy), 128)
        xs.append(xy)
        ds.append(de)
        off.append(off[-1] + len(xy))
    np.save(pxy, np.concatenate(xs))
    np.save(pde, np.concatenate(ds))
    np.save(poff, np.array(off, np.int64))
    print(f"[sift_pack] packed {len(data.meta)} images ({off[-1]} keypoints) in {time.time() - t0:.1f}s")


class Sift:
    def __init__(self):
        pxy, pde, poff = _sift_pack_paths()
        self.xy = np.load(pxy, mmap_mode="r")
        self.desc = np.load(pde, mmap_mode="r")
        self.off = np.load(poff)

    def __getitem__(self, row):
        a, b = self.off[row], self.off[row + 1]
        return np.asarray(self.xy[a:b]), np.asarray(self.desc[a:b])

    def n_kp(self):
        return np.diff(self.off)

In [ ]:
SHOT_BINS = [0, 1, 2, 5, 10, 20, 10 ** 6]


@dataclass
class FoldData:
    species: str
    gallery_rows: np.ndarray
    gallery_labels: np.ndarray
    query_rows: np.ndarray
    query_ids: np.ndarray
    extra: dict = field(default_factory=dict)


@dataclass
class FoldTruth:
    query_ids: np.ndarray
    labels: np.ndarray

    def as_dict(self):
        return dict(zip(self.query_ids, self.labels))


def l2n(a):
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)


def encounter_clusters(U, labels, thr):
    out = np.empty(len(labels), dtype=object)
    for ident in np.unique(labels):
        idx = np.where(labels == ident)[0]
        if len(idx) == 1:
            out[idx] = f"{ident}#0"
            continue
        Z = U[idx]
        D = np.clip(1.0 - Z @ Z.T, 0.0, 2.0)
        np.fill_diagonal(D, 0.0)
        lab = fcluster(linkage(squareform(D, checks=False), "average"), thr, "distance")
        out[idx] = [f"{ident}#{k}" for k in lab]
    return out


def make_split(labels, clusters, seed, p_move, f_new_id, max_new_id_size, species):
    rng = np.random.default_rng(seed * 7919 + SPECIES.index(species))
    s = pd.Series(labels)
    size = s.value_counts()
    elig = size[size <= max_new_id_size]
    bins = pd.cut(elig, SHOT_BINS)
    new_ids = []
    for _, grp in elig.groupby(bins, observed=True):
        k = int(round(f_new_id * len(grp)))
        if k:
            new_ids += list(rng.choice(grp.index.values, k, replace=False))
    new_ids = set(new_ids)
    is_query = np.zeros(len(labels), bool)
    is_query[s.isin(new_ids).values] = True
    cl = pd.Series(clusters)
    for ident in size.index:
        if ident in new_ids:
            continue
        idx = np.where(labels == ident)[0]
        ucl = np.unique(cl.values[idx])
        if len(ucl) < 2:
            continue
        move = rng.random(len(ucl)) < p_move
        if move.all():
            move[rng.integers(len(ucl))] = False
        moved = set(ucl[move])
        is_query[idx[np.isin(cl.values[idx], list(moved))]] = True
    truth = np.where(s.isin(new_ids).values, NEW, labels)
    return is_query, truth


def build_splits(data, cv_cfg, U=None):
    key = json.dumps({k: cv_cfg[k] for k in ("seeds", "enc_thr", "p_move", "f_new_id", "max_new_id_size")}, sort_keys=True)
    h = hashlib.md5(key.encode()).hexdigest()[:10]
    path = CACHE_DIR / "splits" / f"splits_{h}.json"
    tr = data.train
    if U is None:
        U = l2n(data.X.astype(np.float64))
    if path.exists():
        raw = json.loads(path.read_text())
    else:
        raw = {"config": json.loads(key), "splits": []}
        clus = {}
        for sp in SPECIES:
            m = (tr.species == sp).values
            thr = cv_cfg["enc_thr"][sp] if isinstance(cv_cfg["enc_thr"], dict) else cv_cfg["enc_thr"]
            clus[sp] = encounter_clusters(U[tr.row.values[m]], tr.identity.values[m], thr)
        for seed in cv_cfg["seeds"]:
            entry = {}
            for sp in SPECIES:
                m = (tr.species == sp).values
                rows = tr.row.values[m]
                labels = tr.identity.values[m]
                cap = cv_cfg["max_new_id_size"][sp] if isinstance(cv_cfg["max_new_id_size"], dict) else cv_cfg["max_new_id_size"]
                isq, truth = make_split(labels, clus[sp], seed, cv_cfg["p_move"][sp], cv_cfg["f_new_id"][sp], cap, sp)
                entry[sp] = {"gallery": rows[~isq].tolist(), "query": rows[isq].tolist(), "truth": truth[isq].tolist()}
            raw["splits"].append(entry)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(raw))
    out = []
    ident = data.meta.identity.values
    ids = data.meta.id.values
    for entry in raw["splits"]:
        e = {}
        for sp in SPECIES:
            g = np.array(entry[sp]["gallery"])
            q = np.array(entry[sp]["query"])
            e[sp] = (FoldData(sp, g, ident[g], q, ids[q]), FoldTruth(ids[q], np.array(entry[sp]["truth"])))
        out.append(e)
    return out


def test_fold(data, sp):
    tr, te = data.train, data.test
    g = tr.row.values[(tr.species == sp).values]
    q = te.row.values[(te.species == sp).values]
    return FoldData(sp, g, data.meta.identity.values[g], q, data.meta.id.values[q])

In [ ]:
def embed_unsupervised(X, meta, kind, **kw):
    if kind == "l2":
        nrm = np.linalg.norm(X.astype(np.float64), axis=1, keepdims=True)
        return (X / np.maximum(nrm, 1e-12)).astype(np.float32)
    if kind == "pcaw":
        d, eps = kw.get("d", 512), kw.get("eps", 1.0)
        out = np.zeros((len(X), d), dtype=np.float32)
        for sp in SPECIES:
            m = (meta.species == sp).values
            Z = l2n(X[m].astype(np.float64))
            Zc = Z - Z.mean(0)
            C = (Zc.T @ Zc) / max(len(Z) - 1, 1)
            ev, V = np.linalg.eigh(C)
            idx = np.argsort(ev)[::-1]
            ev = np.maximum(ev[idx], 0.0)
            Vd = V[:, idx[:d]]
            out[m] = l2n(Zc @ (Vd / np.sqrt(ev[:d] + eps * ev.mean()))).astype(np.float32)
        return out
    raise ValueError(kind)


def fit_lda(Zg, labels, gamma=0.2, d_out=256, pca_d=512, sides=None, w_cross=0.0):
    mu = Zg.mean(0)
    Zc = (Zg - mu).astype(np.float64)
    C = (Zc.T @ Zc) / max(len(Zc) - 1, 1)
    _, V = np.linalg.eigh(C)
    Vp = V[:, ::-1][:, :pca_d]
    Y = Zc @ Vp
    uniq, inv, cnt = np.unique(labels, return_inverse=True, return_counts=True)
    means = np.zeros((len(uniq), Y.shape[1]))
    np.add.at(means, inv, Y)
    means /= cnt[:, None]
    multi = cnt[inv] >= 2
    R = Y[multi] - means[inv[multi]]
    Sw = R.T @ R / max(len(R) - 1, 1)
    if sides is not None and w_cross > 0:
        D = []
        for k in range(len(uniq)):
            m = inv == k
            L = m & (sides == "L")
            Rr = m & (sides == "R")
            if L.any() and Rr.any():
                D.append(Y[L].mean(0) - Y[Rr].mean(0))
        if len(D) >= 10:
            D = np.array(D)
            Sx = D.T @ D / len(D)
            Sw = Sw / np.trace(Sw) + w_cross * Sx / np.trace(Sx)
    Sw = (1 - gamma) * Sw + gamma * np.trace(Sw) / Sw.shape[0] * np.eye(Sw.shape[0])
    ew, Ew = np.linalg.eigh(Sw)
    Wh = Ew / np.sqrt(np.maximum(ew, 1e-12))
    Mb = (means - means.mean(0)) @ Wh
    Sb = (Mb * cnt[:, None]).T @ Mb / cnt.sum()
    _, Eb = np.linalg.eigh(Sb)
    Eb = Eb[:, ::-1][:, :d_out]
    W = Vp @ Wh @ Eb
    return lambda Z: l2n((Z - mu) @ W)


class IdentityIndex:
    def __init__(self, gallery_labels):
        self.labels = np.asarray(gallery_labels)
        self.ids, inv = np.unique(self.labels, return_inverse=True)
        self.order = np.argsort(inv, kind="stable")
        self.inv = inv
        self.counts = np.bincount(inv)
        self.starts = np.r_[0, np.cumsum(self.counts)[:-1]]

    def max(self, S):
        return np.maximum.reduceat(S[:, self.order], self.starts, axis=1)

    def sorted_desc_k(self, S, k):
        out = np.zeros((S.shape[0], len(self.ids)))
        So = S[:, self.order]
        for j, (a, c) in enumerate(zip(self.starts, self.counts)):
            blk = So[:, a:a + c]
            kk = min(k, c)
            out[:, j] = -np.sort(-blk, axis=1)[:, :kk].mean(1) if c > 1 else blk[:, 0]
        return out

    def prototypes(self, Zg):
        P = np.zeros((len(self.ids), Zg.shape[1]))
        np.add.at(P, self.inv, Zg)
        return l2n(P)


def rg_diff(Z, labels, K=10):
    Sgg = Z @ Z.T
    Sgg_diff = np.where(labels[:, None] == labels[None, :], -2.0, Sgg)
    k = min(K, max(1, len(Z) - 1))
    return np.partition(Sgg_diff, -k, axis=1)[:, -k:].mean(1).astype(np.float32)


def _sw_basis(Zg, m0, labels, reg=0.5):
    names, inv, cnt = np.unique(labels, return_inverse=True, return_counts=True)
    means = np.zeros((len(names), Zg.shape[1]), dtype=np.float64)
    np.add.at(means, inv, Zg)
    means /= cnt[:, None]
    multi = cnt[inv] >= 2
    Zw = (Zg[multi] - means[inv[multi]]).astype(np.float64)
    Sw = (Zw.T @ Zw) / max(len(Zw), 1)
    ew, vw = np.linalg.eigh(Sw)
    ew = np.maximum(ew, 0.0)
    scale = (1.0 / np.sqrt((1.0 - reg) * ew + reg * max(ew.mean(), 1e-12))).astype(np.float32)
    return vw.astype(np.float32), scale, means, cnt


def fit_sw_and_fullda(Zg, glab, reg=0.5, d_lda=512, gside=None):
    Zg = np.asarray(Zg, dtype=np.float32)
    m0 = Zg.mean(0).astype(np.float32)
    vw, scale, means, cnt = _sw_basis(Zg, m0, glab, reg=reg)
    W_sw = vw * scale[None, :]
    Mc_w = ((means - means.mean(0)) @ vw.astype(np.float64)) * scale.astype(np.float64)[None, :]
    Sb_w = (Mc_w * cnt[:, None]).T @ Mc_w / max(cnt.sum(), 1)
    _, Eb = np.linalg.eigh(Sb_w)
    W_fl = W_sw @ Eb[:, ::-1][:, :d_lda].astype(np.float32)
    out = {
        "cl2": lambda Z, m0=m0: l2n(Z - m0).astype(np.float32),
        "sw": lambda Z, m0=m0, W=W_sw: l2n((Z - m0) @ W).astype(np.float32),
        "fl512": lambda Z, m0=m0, W=W_fl: l2n((Z - m0) @ W).astype(np.float32),
    }
    if gside is not None:
        glab_side = np.array([f"{a}|{b}" if b in ("L", "R") else a for a, b in zip(glab, gside)])
        vws, scs, _, _ = _sw_basis(Zg, m0, glab_side, reg=reg)
        W_ssw = vws * scs[None, :]
        out["ssw"] = lambda Z, m0=m0, W=W_ssw: l2n((Z - m0) @ W).astype(np.float32)
    return out


def vp_compat_stats(Zg, labels, vps, min_n=8):
    acc = {}
    for _, idx in pd.Series(np.arange(len(labels))).groupby(labels):
        idx = idx.values
        if len(idx) < 2:
            continue
        S = Zg[idx] @ Zg[idx].T
        iu = np.triu_indices(len(idx), 1)
        for a, b, x in zip(vps[idx][iu[0]], vps[idx][iu[1]], S[iu]):
            k = (a, b) if a <= b else (b, a)
            acc.setdefault(k, []).append(float(x))
    return {k: (float(np.mean(v)), float(max(np.std(v), 0.05))) for k, v in acc.items() if len(v) >= min_n}

In [ ]:
LNB_KNN = 50
LNB_K = 10
_LNB_SCORER_CACHE = {}
_LNB_FOLD_CACHE = {}
_LNB_UIMG_CACHE = {}


def _numpy_knn_l2(Q, X, k):
    # Fallback exact L2^2 kNN using BLAS matmul when faiss is unavailable
    x2 = (X * X).sum(1)[None, :]
    q2 = (Q * Q).sum(1)[:, None]
    d2 = np.maximum(q2 + x2 - 2.0 * (Q @ X.T), 0.0)
    idx = np.argpartition(d2, k, axis=1)[:, :k]
    vals = np.take_along_axis(d2, idx, axis=1)
    ord_ = np.argsort(vals, axis=1)
    return np.take_along_axis(vals, ord_, axis=1), np.take_along_axis(idx, ord_, axis=1)


def ensure_lnbnn_cache(data: Data, out_dir: Path, ef_search=64):
    out_dir = Path(out_dir)
    if all((out_dir / f"{sp}_meta.json").exists() and (out_dir / f"{sp}_nb.npy").exists() and (out_dir / f"{sp}_d.npy").exists() for sp in SPECIES):
        print(f"[lnbnn] using cached {out_dir}")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    S = Sift()
    row = dict(zip(data.meta.id, data.meta.row))

    def _rs(i):
        d = S[row[i]][1].astype(np.float32).reshape(-1, 128)
        return np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1e-7))

    tr, te = data.train, data.test
    for sp in SPECIES:
        t0 = time.time()
        gids = tr.id[tr.species == sp].values
        qids = np.r_[gids, te.id[te.species == sp].values]
        X_list = [_rs(g) for g in gids]
        owner = np.repeat(np.arange(len(gids)), [len(x) for x in X_list]).astype(np.int32)
        X = np.concatenate(X_list)
        if HAS_FAISS:
            index = faiss.IndexHNSWFlat(128, 32)
            index.hnsw.efConstruction = 64
            index.add(X)
            index.hnsw.efSearch = ef_search
            print(f"[lnbnn] {sp}: HNSW index built ({len(X)} descs) in {time.time() - t0:.1f}s", flush=True)
        else:
            index = None
            print(f"[lnbnn] {sp}: using NumPy exact search ({len(X)} descs)", flush=True)
        qlen = [len(S[row[q]][1]) for q in qids]
        qoff = np.r_[0, np.cumsum(qlen)]
        NB = np.lib.format.open_memmap(out_dir / f"{sp}_nb.npy", "w+", np.int32, (int(qoff[-1]), LNB_KNN))
        DI = np.lib.format.open_memmap(out_dir / f"{sp}_d.npy", "w+", np.float16, (int(qoff[-1]), LNB_KNN))
        bs = 400 if HAS_FAISS else 64
        for b in range(0, len(qids), bs):
            e = min(b + bs, len(qids))
            if qoff[e] == qoff[b]:
                continue
            Q = np.concatenate([_rs(q) for q in qids[b:e]])
            if HAS_FAISS:
                d, i = index.search(Q, LNB_KNN)
            else:
                d, i = _numpy_knn_l2(Q, X, LNB_KNN)
            NB[qoff[b]:qoff[e]] = np.where(i >= 0, owner[np.maximum(i, 0)], -1)
            DI[qoff[b]:qoff[e]] = np.sqrt(np.maximum(d, 0.0))
        NB.flush()
        DI.flush()
        (out_dir / f"{sp}_meta.json").write_text(json.dumps({"gallery": gids.tolist(), "queries": qids.tolist(), "qoff": qoff.tolist()}))
        print(f"[lnbnn] {sp}: {len(qids)} query images finished in {time.time() - t0:.1f}s", flush=True)


class Scorer:
    def __init__(self, d: Path, sp: str, train_ids, train_labels):
        m = json.loads((d / f"{sp}_meta.json").read_text())
        self.gal = np.array(m["gallery"])
        self.qpos = {q: i for i, q in enumerate(m["queries"])}
        self.qoff = np.array(m["qoff"])
        self.NB = np.load(d / f"{sp}_nb.npy", mmap_mode="r")
        self.DI = np.load(d / f"{sp}_d.npy", mmap_mode="r")
        lab = dict(zip(train_ids, train_labels))
        self.names, self.lab = np.unique([lab[g] for g in self.gal], return_inverse=True)
        self.gpos = {g: i for i, g in enumerate(self.gal)}

    def scores_all(self, qid, gmask, ratio=0.85):
        i = self.qpos[qid]
        a, b = self.qoff[i], self.qoff[i + 1]
        n_id, n_gal = len(self.names), len(self.gal)
        out_k10 = np.zeros(n_id, np.float32)
        out_img = np.zeros(n_id, np.float32)
        out_k3 = np.zeros(n_id, np.float32)
        out_uimg = np.zeros(n_id, np.float32)
        out_rimg = np.zeros(n_id, np.float32)
        img_sc = np.zeros(n_gal, np.float32)
        img_u = np.zeros(n_gal, np.float32)
        if b == a:
            return out_k10, out_img, out_k3, out_uimg, out_rimg, img_sc, img_u
        nb = np.asarray(self.NB[a:b])
        d = np.asarray(self.DI[a:b], np.float32)
        ok = (nb >= 0) & gmask[np.maximum(nb, 0)]
        if qid in self.gpos:
            ok &= nb != self.gpos[qid]
        rank = np.cumsum(ok, 1)
        rows_all = np.arange(len(d))
        lab_mat = self.lab[np.maximum(nb, 0)]

        has_bg10 = rank[:, -1] >= LNB_K + 1
        bg10 = np.where(has_bg10, d[rows_all, np.argmax(rank >= LNB_K + 1, 1)], d.max(1))
        r10, c10 = np.nonzero(ok & (rank <= LNB_K))
        if len(r10):
            l10 = lab_mat[r10, c10]
            _, f_id = np.unique(r10.astype(np.int64) * n_id + l10, return_index=True)
            np.add.at(out_k10, l10[f_id], np.maximum(bg10[r10[f_id]] - d[r10[f_id], c10[f_id]], 0.0))
            g10 = nb[r10, c10]
            _, f_g = np.unique(r10.astype(np.int64) * n_gal + g10, return_index=True)
            np.add.at(img_sc, g10[f_g], np.maximum(bg10[r10[f_g]] - d[r10[f_g], c10[f_g]], 0.0))
            np.maximum.at(out_img, self.lab, img_sc)

        has_bg3 = rank[:, -1] >= 4
        bg3 = np.where(has_bg3, d[rows_all, np.argmax(rank >= 4, 1)], d.max(1))
        r3, c3 = np.nonzero(ok & (rank <= 3))
        if len(r3):
            l3 = lab_mat[r3, c3]
            _, f3 = np.unique(r3.astype(np.int64) * n_id + l3, return_index=True)
            np.add.at(out_k3, l3[f3], np.maximum(bg3[r3[f3]] - d[r3[f3], c3[f3]], 0.0))

        has1 = rank[:, -1] >= 1
        c1 = np.argmax(ok, 1)
        g1 = nb[rows_all, c1]
        id1 = np.where(has1, self.lab[np.maximum(g1, 0)], -1)
        ok_diff = ok & (lab_mat != id1[:, None])
        has2 = ok_diff.any(1)
        d2 = np.where(has2, d[rows_all, np.argmax(ok_diff, 1)], d.max(1))
        ok_same = ok & (lab_mat == id1[:, None]) & (d < d2[:, None])
        rs, cs = np.nonzero(ok_same)
        if len(rs):
            gs = nb[rs, cs]
            _, fg = np.unique(rs.astype(np.int64) * n_gal + gs, return_index=True)
            rs1, cs1, gs1 = rs[fg], cs[fg], gs[fg]
            vm = np.maximum(d2[rs1] - d[rs1, cs1], 0.0)
            np.add.at(img_u, gs1, vm)
            np.maximum.at(out_uimg, self.lab, img_u)
            if has2.any():
                vr = np.where(has2[rs1] & (d[rs1, cs1] < ratio * d2[rs1]), vm, 0.0)
                img_r = np.zeros(n_gal, np.float32)
                np.add.at(img_r, gs1, vr)
                np.maximum.at(out_rimg, self.lab, img_r)
        return out_k10, out_img, out_k3, out_uimg, out_rimg, img_sc, img_u


def get_scorer(ctx, sp):
    d = Path(ctx.cfg["lnbnn_dir"])
    key = (str(d), sp)
    if key not in _LNB_SCORER_CACHE:
        tr = ctx.data.train
        _LNB_SCORER_CACHE[key] = Scorer(d, sp, tr.id.values, tr.identity.values)
    return _LNB_SCORER_CACHE[key]


def fold_all_scores(fold, ctx, cand_ids):
    d = ctx.cfg["lnbnn_dir"]
    key = (d, fold.species, hash(fold.query_rows.tobytes()), hash(fold.gallery_rows.tobytes()))
    if key in _LNB_FOLD_CACHE:
        return _LNB_FOLD_CACHE[key]
    S = get_scorer(ctx, fold.species)
    gids = ctx.data.meta.id.values[fold.gallery_rows]
    gmask = np.isin(S.gal, gids)
    col_idx = np.searchsorted(S.names, cand_ids)
    g_col = np.array([S.gpos[g] for g in gids], dtype=np.int32)
    res = [S.scores_all(q, gmask) for q in fold.query_ids]
    L = np.stack([x[0] for x in res])[:, col_idx]
    Lim = np.stack([x[1] for x in res])[:, col_idx]
    Lk3 = np.stack([x[2] for x in res])[:, col_idx]
    Lu = np.stack([x[3] for x in res])[:, col_idx]
    Lr = np.stack([x[4] for x in res])[:, col_idx]
    ImgMat = np.stack([x[5][g_col] for x in res])
    UimgMat = np.stack([x[6][g_col] for x in res])
    _LNB_FOLD_CACHE[key] = (L, Lim, Lk3, Lu, Lr, ImgMat)
    _LNB_UIMG_CACHE[key] = UimgMat
    return _LNB_FOLD_CACHE[key]


def fold_uimg_scores(fold, ctx, cand_ids):
    _, Lim, _, Lu, _, ImgMat = fold_all_scores(fold, ctx, cand_ids)
    d = ctx.cfg["lnbnn_dir"]
    key = (d, fold.species, hash(fold.query_rows.tobytes()), hash(fold.gallery_rows.tobytes()))
    return Lu, Lim, ImgMat, _LNB_UIMG_CACHE[key]


def add_lnbnn_features(df, fold, ctx, cand_ids):
    L, Lim, Lk3, Lu, _, _ = fold_all_scores(fold, ctx, cand_ids)
    mx = L.max(1)
    second = -np.sort(-L, 1)[:, 1] if L.shape[1] > 1 else mx
    rk = np.argsort(np.argsort(-L, 1), 1)
    qi, c = df.qi.values, df.cand.values
    v = L[qi, c]
    df["lnb"] = v
    df["lnb_rank"] = rk[qi, c]
    df["lnb_rel"] = np.where(mx[qi] > 0, v / np.maximum(mx[qi], 1e-9), 0.0)
    df["lnb_log"] = np.log1p(v)
    df["q_lnb_max"] = mx[qi]
    df["q_lnb_gap"] = (mx - second)[qi]
    df["q_lnb_log_max"] = np.log1p(mx)[qi]
    if ctx.cfg["retrieval"].get("lnbnn_img", False):
        mx_i = Lim.max(1)
        sec_i = -np.sort(-Lim, 1)[:, 1] if Lim.shape[1] > 1 else mx_i
        rk_i = np.argsort(np.argsort(-Lim, 1), 1)
        vi = Lim[qi, c]
        df["lnb_img"] = vi
        df["lnb_img_rank"] = rk_i[qi, c]
        df["lnb_img_rel"] = np.where(mx_i[qi] > 0, vi / np.maximum(mx_i[qi], 1e-9), 0.0)
        df["lnb_img_log"] = np.log1p(vi)
        df["lnb_img_share"] = np.where(v > 0, vi / np.maximum(v, 1e-6), 0.0)
        df["q_lnb_img_max"] = mx_i[qi]
        df["q_lnb_img_gap"] = (mx_i - sec_i)[qi]
        df["q_lnb_img_log_max"] = np.log1p(mx_i)[qi]
        mx_k = Lk3.max(1)
        sec_k = -np.sort(-Lk3, 1)[:, 1] if Lk3.shape[1] > 1 else mx_k
        vk = Lk3[qi, c]
        df["lnb_k3"] = vk
        df["lnb_k3_rel"] = np.where(mx_k[qi] > 0, vk / np.maximum(mx_k[qi], 1e-9), 0.0)
        df["q_lnb_k3_max"] = mx_k[qi]
        df["q_lnb_k3_gap"] = (mx_k - sec_k)[qi]
    if ctx.cfg["retrieval"].get("bg_clean", False):
        mx_u = Lu.max(1)
        sec_u = -np.sort(-Lu, 1)[:, 1] if Lu.shape[1] > 1 else mx_u
        rk_u = np.argsort(np.argsort(-Lu, 1), 1)
        vu = Lu[qi, c]
        df["lnb_uimg"] = vu
        df["lnb_uimg_rank"] = rk_u[qi, c]
        df["lnb_uimg_rel"] = np.where(mx_u[qi] > 0, vu / np.maximum(mx_u[qi], 1e-9), 0.0)
        df["lnb_uimg_log"] = np.log1p(vu)
        df["q_lnb_uimg_max"] = mx_u[qi]
        df["q_lnb_uimg_gap"] = (mx_u - sec_u)[qi]
        df["q_lnb_uimg_log_max"] = np.log1p(mx_u)[qi]
    return df

In [ ]:
SF = ["kp_min", "kp_max", "n_ratio", "n_mutual", "inl_aff", "inl_h", "inl_ratio", "inl_over_kp",
      "inl_sim", "log_scale", "abs_rot", "spread_min", "spread_max", "inl_aff_strict"]
NF = len(SF)


def rootsift(desc):
    d = desc.astype(np.float32)
    d /= d.sum(1, keepdims=True) + 1e-7
    return np.sqrt(d)


def _ratio_nn(Sim, ratio):
    if Sim.shape[1] < 2:
        return np.zeros(Sim.shape[0], int), np.zeros(Sim.shape[0], bool)
    part = np.argpartition(-Sim, 1, axis=1)[:, :2]
    s = np.take_along_axis(Sim, part, 1)
    swap = s[:, 1] > s[:, 0]
    best = np.where(swap, part[:, 1], part[:, 0])
    s1 = np.maximum(s.max(1), -1)
    s2 = s.min(1)
    d1 = np.sqrt(np.maximum(2 - 2 * s1, 0))
    d2 = np.sqrt(np.maximum(2 - 2 * s2, 0))
    return best, d1 < ratio * d2


def _spread(pts, allpts):
    sa = allpts.std(0).mean()
    return float(pts.std(0).mean() / sa) if sa > 0 else np.nan


def match_features(xa, ra, xb, rb, ratio=0.8, ransac_frac=0.02):
    f = np.full(NF, np.nan, np.float32)
    na, nb = len(xa), len(xb)
    f[0], f[1] = min(na, nb), max(na, nb)
    if na < 2 or nb < 2:
        return f
    Sim = ra @ rb.T
    ba, oka = _ratio_nn(Sim, ratio)
    bb, okb = _ratio_nn(np.ascontiguousarray(Sim.T), ratio)
    f[2] = oka.sum() + okb.sum()
    ia = np.where(oka & okb[ba] & (bb[ba] == np.arange(na)))[0]
    ib = ba[ia]
    f[3] = len(ia)
    f[4:6] = 0
    if len(ia) < 4:
        return f
    pa = xa[ia].astype(np.float32)
    pb = xb[ib].astype(np.float32)
    side = float(max(xa.max(), xb.max(), 1.0))
    thr = ransac_frac * side
    M, inl = cv2.estimateAffinePartial2D(pa, pb, method=cv2.RANSAC, ransacReprojThreshold=thr, maxIters=2000, confidence=0.995)
    if M is not None and inl is not None:
        inl = inl.ravel().astype(bool)
        k = int(inl.sum())
        f[4] = k
        f[6] = k / len(ia)
        f[7] = k / min(na, nb)
        if k:
            f[8] = float(Sim[ia[inl], ib[inl]].mean())
        sc = float(np.hypot(M[0, 0], M[1, 0]))
        f[9] = np.log(max(sc, 1e-6))
        f[10] = abs(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
        if k >= 3:
            s1, s2 = _spread(pa[inl], xa), _spread(pb[inl], xb)
            f[11], f[12] = min(s1, s2), max(s1, s2)
        f[13] = k if (0.5 <= sc <= 2.0 and f[10] < 45) else 0
    H, inlh = cv2.findHomography(pa, pb, cv2.RANSAC, thr)
    if H is not None and inlh is not None:
        f[5] = int(inlh.sum())
    return f


_FORK_SIFT = None
_FORK_RS = {}
_FORK_RATIO = 0.8
_FORK_FRAC = 0.02


def _fork_get_rs(idx):
    v = _FORK_RS.get(idx)
    if v is None:
        if len(_FORK_RS) >= 500:
            _FORK_RS.clear()
        x, d = _FORK_SIFT[idx]
        v = (x, rootsift(d) if len(d) else d.astype(np.float32))
        _FORK_RS[idx] = v
    return v


def _fork_work(group):
    a, bs = group
    xa, ra = _fork_get_rs(a)
    out = np.zeros((len(bs), NF), np.float32)
    for j, b in enumerate(bs):
        xb, rb = _fork_get_rs(b)
        out[j] = match_features(xa, ra, xb, rb, _FORK_RATIO, _FORK_FRAC)
    return a, bs, out


class PairCache:
    # Memory-efficient sorted-array SIFT pair cache (~300 MB for 4.8M pairs vs ~2 GB dict)

    def __init__(self, sift_cfg):
        self.cfg = sift_cfg
        tag = f"r{sift_cfg['ratio']}_f{sift_cfg['ransac_frac']}"
        self.work_path = CACHE_DIR / f"sift_pairs_{tag}.npz"
        self.in_path = PRE_SIFT_PAIRS if (PRE_SIFT_PAIRS is not None and PRE_SIFT_PAIRS.name == self.work_path.name) else None
        self._arr_keys = np.zeros(0, dtype=np.int64)
        self._arr_feats = np.zeros((0, NF), dtype=np.float32)
        self._extra = {}
        if self.in_path is not None and self.in_path.resolve() != self.work_path.resolve():
            self._load_npz(self.in_path)
        self._load_npz(self.work_path)
        if len(self._arr_keys):
            print(f"[sift_cache] preloaded {len(self._arr_keys):,} cached SIFT pairs")

    @staticmethod
    def _pack(a, b):
        return (int(a) << 20) | int(b) if a < b else (int(b) << 20) | int(a)

    @staticmethod
    def _pack_arr(keys_2d):
        a = keys_2d[:, 0].astype(np.int64)
        b = keys_2d[:, 1].astype(np.int64)
        lo = np.minimum(a, b)
        hi = np.maximum(a, b)
        return (lo << 20) | hi

    def _merge_sorted(self, new_pk, new_feats):
        if len(new_pk) == 0:
            return
        if len(self._arr_keys) == 0:
            pk, feats = new_pk, new_feats
        else:
            pk = np.concatenate([self._arr_keys, new_pk])
            feats = np.concatenate([self._arr_feats, new_feats])
        ord_ = np.argsort(pk, kind="mergesort")
        pk, feats = pk[ord_], feats[ord_]
        uniq_mask = np.r_[True, pk[1:] != pk[:-1]]
        self._arr_keys = np.ascontiguousarray(pk[uniq_mask])
        self._arr_feats = np.ascontiguousarray(feats[uniq_mask])

    def _load_npz(self, path):
        if path is not None and path.exists():
            z = np.load(path)
            if len(z["keys"]):
                self._merge_sorted(self._pack_arr(z["keys"]), z["feats"].astype(np.float32))

    @staticmethod
    def key(a, b):
        return (a, b) if a < b else (b, a)

    def _flush_extra(self):
        if not self._extra:
            return
        pk = np.fromiter(self._extra.keys(), dtype=np.int64, count=len(self._extra))
        feats = np.stack(list(self._extra.values())).astype(np.float32)
        self._extra.clear()
        self._merge_sorted(pk, feats)

    def ensure(self, pairs, workers=None, verbose=True):
        if not pairs:
            return 0
        self._flush_extra()
        arr = np.asarray(pairs, dtype=np.int64)
        arr = arr[arr[:, 0] != arr[:, 1]]
        if not len(arr):
            return 0
        pks = np.unique(self._pack_arr(arr))
        if len(self._arr_keys):
            pos = np.searchsorted(self._arr_keys, pks)
            missing_mask = (pos >= len(self._arr_keys)) | (self._arr_keys[np.minimum(pos, len(self._arr_keys) - 1)] != pks)
            need_pks = pks[missing_mask]
        else:
            need_pks = pks
        if not len(need_pks):
            return 0

        groups = {}
        for pk in need_pks:
            a, b = int(pk >> 20), int(pk & 0xFFFFF)
            groups.setdefault(a, []).append(b)
        items = sorted(groups.items(), key=lambda kv: -len(kv[1]))
        t0 = time.time()
        workers = workers or self.cfg["workers"] or N_THREADS
        done = 0
        next_log = 50_000
        next_save = 200_000

        global _FORK_SIFT, _FORK_RS, _FORK_RATIO, _FORK_FRAC
        _FORK_SIFT = Sift()
        _FORK_RS = {}
        _FORK_RATIO = float(self.cfg["ratio"])
        _FORK_FRAC = float(self.cfg["ransac_frac"])

        if hasattr(os, "fork") and workers > 1:
            ctx_mp = mp.get_context("fork")
            with ctx_mp.Pool(processes=workers) as pool:
                for a, bs, feats in pool.imap_unordered(_fork_work, items, chunksize=8):
                    for b, f in zip(bs, feats):
                        self._extra[(a << 20) | b] = f
                    done += len(bs)
                    if verbose and done >= next_log:
                        dt = time.time() - t0
                        eta = dt * (len(need_pks) - done) / max(done, 1)
                        print(f"  [sift] {done:,}/{len(need_pks):,} ({1000 * dt / done:.2f} ms/pair, {dt:.0f}s elapsed, ETA {eta:.0f}s)", flush=True)
                        next_log += 50_000
                    if done >= next_save:
                        self.save()
                        next_save += 200_000
        else:
            with ThreadPoolExecutor(max_workers=workers) as ex:
                for a, bs, feats in ex.map(_fork_work, items):
                    for b, f in zip(bs, feats):
                        self._extra[(a << 20) | b] = f
                    done += len(bs)
                    if verbose and done >= next_log:
                        dt = time.time() - t0
                        eta = dt * (len(need_pks) - done) / max(done, 1)
                        print(f"  [sift] {done:,}/{len(need_pks):,} ({1000 * dt / done:.2f} ms/pair, {dt:.0f}s elapsed, ETA {eta:.0f}s)", flush=True)
                        next_log += 50_000
                    if done >= next_save:
                        self.save()
                        next_save += 200_000

        if verbose:
            dt = time.time() - t0
            print(f"  [sift] computed {len(need_pks):,} pairs in {dt:.0f}s ({1000 * dt / len(need_pks):.2f} ms/pair wall)", flush=True)
        self.save()
        return len(need_pks)

    def save(self):
        self._flush_extra()
        keys = np.column_stack([(self._arr_keys >> 20).astype(np.int32), (self._arr_keys & 0xFFFFF).astype(np.int32)])
        tmp = self.work_path.with_suffix(".tmp.npz")
        np.savez(tmp, keys=keys, feats=self._arr_feats)
        os.replace(tmp, self.work_path)

    def get(self, a, b):
        if a == b:
            return None
        pk = self._pack(a, b)
        v = self._extra.get(pk)
        if v is not None:
            return v
        if len(self._arr_keys):
            pos = int(np.searchsorted(self._arr_keys, pk))
            if pos < len(self._arr_keys) and self._arr_keys[pos] == pk:
                return self._arr_feats[pos]
        return None

In [ ]:
SIDE = {"left": "L", "frontleft": "L", "backleft": "L", "right": "R", "frontright": "R", "backright": "R"}


def side_of(v):
    return SIDE.get(v, "N") if isinstance(v, str) else "U"


class Context:
    def __init__(self, data, emb, kp, cfg, pair_cache=None):
        self.data, self.emb, self.kp, self.cfg, self.pc = data, emb, kp, cfg, pair_cache
        vp = data.meta.viewpoint.copy()
        self.vp = vp.fillna("NaN").values
        self.side = np.array([side_of(v) for v in vp.values])
        self.lognorm = np.log(np.linalg.norm(data.X, axis=1).astype(np.float64))
        self._v25_cache = {}


def vp_compat(Zg, labels, vps):
    acc = {}
    for _, idx in pd.Series(np.arange(len(labels))).groupby(labels):
        idx = idx.values
        if len(idx) < 2:
            continue
        S = Zg[idx] @ Zg[idx].T
        iu = np.triu_indices(len(idx), 1)
        for a, b, x in zip(vps[idx][iu[0]], vps[idx][iu[1]], S[iu]):
            k = (a, b) if a <= b else (b, a)
            t = acc.setdefault(k, [0.0, 0])
            t[0] += x
            t[1] += 1
    return {k: sm / n for k, (sm, n) in acc.items() if n >= 10}


def manifold_diffusion(Zq, Zg, k=20, alpha=0.85, gamma=3.0, iters=25):
    nq = len(Zq)
    Z = np.vstack([Zq, Zg]).astype(np.float32)
    N = len(Z)
    S = np.clip(Z @ Z.T, 0.0, 1.0)
    np.fill_diagonal(S, 0.0)
    kk = min(k, N - 1)
    top = np.argpartition(-S, kk, axis=1)[:, :kk]
    rows = np.repeat(np.arange(N), kk)
    cols = top.ravel()
    vals = (S[rows, cols] ** gamma).astype(np.float32)
    W = sp.coo_matrix((vals, (rows, cols)), shape=(N, N)).tocsr()
    W = 0.5 * (W + W.T)
    d = np.asarray(W.sum(1)).ravel()
    dinv = np.where(d > 0, 1.0 / np.sqrt(np.maximum(d, 1e-12)), 0.0)
    Sn = sp.diags(dinv) @ W @ sp.diags(dinv)
    Y = np.zeros((N, nq), dtype=np.float32)
    Y[np.arange(nq), np.arange(nq)] = 1.0
    F = Y.copy()
    for _ in range(iters):
        F = alpha * (Sn @ F) + (1.0 - alpha) * Y
    return F[nq:, :].T


def k_reciprocal_rerank(Zq, Zg, k1=20, k2=6, lam=0.3):
    nq, ng = len(Zq), len(Zg)
    Z = np.vstack([Zq, Zg]).astype(np.float32)
    N = len(Z)
    orig_dist = np.clip(2.0 - 2.0 * (Z @ Z.T), 0.0, 4.0)
    orig_dist = orig_dist / np.maximum(orig_dist.max(axis=1, keepdims=True), 1e-6)
    initial_rank = np.argsort(orig_dist, axis=1).astype(np.int32)
    V = np.zeros((N, N), dtype=np.float32)
    hk = int(round(k1 / 2)) + 1
    for i in range(N):
        fwd = initial_rank[i, :k1 + 1]
        bwd = initial_rank[fwd, :k1 + 1]
        k_recip = fwd[np.where(bwd == i)[0]]
        k_recip_exp = k_recip
        for cand in k_recip:
            cfwd = initial_rank[cand, :hk]
            cbwd = initial_rank[cfwd, :hk]
            c_recip = cfwd[np.where(cbwd == cand)[0]]
            if len(np.intersect1d(c_recip, k_recip)) > (2.0 / 3.0) * len(c_recip):
                k_recip_exp = np.append(k_recip_exp, c_recip)
        k_recip_exp = np.unique(k_recip_exp)
        w = np.exp(-orig_dist[i, k_recip_exp])
        V[i, k_recip_exp] = w / np.maximum(w.sum(), 1e-6)
    if k2 > 1:
        V = np.array([V[initial_rank[i, :k2]].mean(axis=0) for i in range(N)], dtype=np.float32)
    Vq, Vg = V[:nq], V[nq:]
    jaccard = np.ones((nq, ng), dtype=np.float32)
    for i in range(nq):
        nz = np.nonzero(Vq[i])[0]
        if len(nz):
            mn = np.minimum(Vq[i, nz][None, :], Vg[:, nz]).sum(axis=1)
            jaccard[i] = 1.0 - mn / (2.0 - mn)
    return 1.0 - ((1.0 - lam) * jaccard + lam * orig_dist[:nq, nq:])


def _fold_v25(fold, ctx, idx, with_diff=False):
    key = (fold.species, hash(fold.gallery_rows.tobytes()), hash(fold.query_rows.tobytes()))
    b = ctx._v25_cache.get(key)
    if b is None:
        rcfg = ctx.cfg["retrieval"]
        qr, gr = fold.query_rows, fold.gallery_rows
        glab = fold.gallery_labels
        projs = fit_sw_and_fullda(ctx.emb["l2"][gr], glab, reg=float(rcfg.get("sw_reg", 0.5)), d_lda=int(rcfg.get("fl_dim", 512)), gside=ctx.side[gr])
        hub_w = float(rcfg.get("sw_hub", 0.5))
        Z_p, S_raw, M_v = {}, {}, {}
        for k in ("cl2", "sw", "ssw", "fl512"):
            zq, zg = projs[k](ctx.emb["l2"][qr]), projs[k](ctx.emb["l2"][gr])
            Z_p[k] = (zq, zg)
            sr = (zq @ zg.T).astype(np.float32)
            S_raw[k] = sr
            if k == "cl2":
                M_v["cl2"] = idx.max(sr)
            else:
                rg = rg_diff(zg, glab, K=10)
                M_v[f"{k}2152" if k in ("sw", "ssw") else k] = idx.max(sr - hub_w * rg[None, :])
                M_v[f"{k}_raw"] = idx.max(sr)
        Kr_sw = k_reciprocal_rerank(Z_p["sw"][0], Z_p["sw"][1], k1=10, k2=4, lam=0.3)
        Mk_sw = idx.max(Kr_sw)
        compat_sw = vp_compat_stats(Z_p["sw"][1], glab, ctx.vp[gr])
        b = {"Z": Z_p, "S_sw_raw": S_raw["sw"], "M": M_v, "Mk_sw": Mk_sw, "compat_sw": compat_sw, "Md_sw": None}
        ctx._v25_cache[key] = b
    if with_diff and b["Md_sw"] is None:
        Dr_sw = manifold_diffusion(b["Z"]["sw"][0], b["Z"]["sw"][1], k=20, alpha=0.85, gamma=3.0, iters=25)
        b["Md_sw"] = idx.max(Dr_sw)
    return b


def sift_pairs_for_fold(fold, ctx):
    cfg = ctx.cfg["sift"]
    rcfg = ctx.cfg["retrieval"]
    n_cand_sift, per_id = cfg["top_ids"], cfg["imgs_per_id"]
    Zq, Zg = ctx.emb["l2"][fold.query_rows], ctx.emb["l2"][fold.gallery_rows]
    S = Zq @ Zg.T
    idx = IdentityIndex(fold.gallery_labels)
    M = idx.max(S)
    M2 = idx.max(ctx.emb["pcaw"][fold.query_rows] @ ctx.emb["pcaw"][fold.gallery_rows].T)
    lnb_k = int(rcfg.get("lnbnn_cand", 0))
    lnb_uk = int(rcfg.get("lnbnn_ucand", 0))
    v25 = bool(rcfg.get("v25_stage1", False))
    Lu, Lim, ImgMat, UimgMat = fold_uimg_scores(fold, ctx, idx.ids) if rcfg.get("lnbnn", False) else (None, None, None, None)
    bv = _fold_v25(fold, ctx, idx, with_diff=False) if v25 else None
    sw_k = int(rcfg.get("sw_cand", 15)) if v25 else 0
    fl_k = int(rcfg.get("fl_cand", 15)) if v25 else 0
    kr_k = int(rcfg.get("krec_sw_cand", 10)) if v25 else 0
    per_id_sw = int(rcfg.get("sift_imgs_sw", 2)) if v25 else 0
    k_im = int(rcfg.get("lnbnn_sift_imgs", 2))
    k_um = int(rcfg.get("lnbnn_uimg_sift_imgs", 1)) if v25 else 0
    pairs = []
    for i in range(len(Zq)):
        cset = np.union1d(np.argsort(-M[i])[:n_cand_sift], np.argsort(-M2[i])[:n_cand_sift])
        if Lim is not None and lnb_k > 0:
            cset = np.union1d(cset, np.argsort(-Lim[i])[:lnb_k])
        if Lu is not None and lnb_uk > 0:
            cset = np.union1d(cset, np.argsort(-Lu[i])[:lnb_uk])
        if bv is not None:
            if sw_k > 0:
                cset = np.union1d(cset, np.argsort(-bv["M"]["sw2152"][i])[:sw_k])
            if fl_k > 0:
                cset = np.union1d(cset, np.argsort(-bv["M"]["fl512"][i])[:fl_k])
            if kr_k > 0:
                cset = np.union1d(cset, np.argsort(-bv["Mk_sw"][i])[:kr_k])
        for c in cset:
            cols = idx.order[idx.starts[c]: idx.starts[c] + idx.counts[c]]
            best = cols[np.argsort(-S[i, cols])[:per_id]]
            if bv is not None and per_id_sw > 0:
                best = np.union1d(best, cols[np.argsort(-bv["S_sw_raw"][i, cols])[:per_id_sw]])
            if ImgMat is not None and k_im > 0:
                best = np.union1d(best, cols[np.argsort(-ImgMat[i, cols])[:k_im]])
            if UimgMat is not None and k_um > 0:
                best = np.union1d(best, cols[np.argsort(-UimgMat[i, cols])[:k_um]])
            pairs += [(fold.query_rows[i], fold.gallery_rows[j]) for j in best]
    return pairs


def query_neighbour_pairs(fold, ctx, k):
    Z = ctx.emb["l2"][fold.query_rows]
    S = Z @ Z.T
    np.fill_diagonal(S, -np.inf)
    nn = np.argsort(-S, axis=1)[:, :k]
    return [(fold.query_rows[i], fold.query_rows[j]) for i in range(len(Z)) for j in nn[i]]


def build_table(fold, ctx, n_cand=30, transductive=True, lda_cfg=None):
    lda_cfg = lda_cfg or {"gamma": 0.3, "d_out": 256}
    gr, qr = fold.gallery_rows, fold.query_rows
    Z = {"l2": (ctx.emb["l2"][qr], ctx.emb["l2"][gr]), "pcaw": (ctx.emb["pcaw"][qr], ctx.emb["pcaw"][gr])}
    f = fit_lda(ctx.emb["l2"][gr], fold.gallery_labels, gamma=lda_cfg["gamma"], d_out=lda_cfg["d_out"])
    Z["lda"] = (f(ctx.emb["l2"][qr]), f(ctx.emb["l2"][gr]))
    EMBS = ("l2", "pcaw", "lda")
    idx = IdentityIndex(fold.gallery_labels)
    nq, nid = len(qr), len(idx.ids)
    S, M, T2, PR = {}, {}, {}, {}
    for e in EMBS:
        zq, zg = Z[e]
        S[e] = zq @ zg.T
        M[e] = idx.max(S[e])
        T2[e] = idx.sorted_desc_k(S[e], 2)
        PR[e] = zq @ idx.prototypes(zg).T
    rank = {e: np.argsort(np.argsort(-M[e], axis=1), axis=1) for e in EMBS}
    comb = sum(rank[e] for e in EMBS)
    gvp, gside = ctx.vp[gr], ctx.side[gr]
    id_of_col = idx.inv
    n_img = idx.counts
    n_vp = pd.Series(gvp).groupby(id_of_col).nunique().reindex(range(nid)).values
    has_L = pd.Series(gside == "L").groupby(id_of_col).any().reindex(range(nid)).values
    has_R = pd.Series(gside == "R").groupby(id_of_col).any().reindex(range(nid)).values
    compat = vp_compat(Z["l2"][1], fold.gallery_labels, gvp)
    pool = np.vstack([Z["l2"][1], Z["l2"][0]]).astype(np.float32)
    GS = np.sort(Z["l2"][1].astype(np.float32) @ pool.T, axis=1)
    npool = GS.shape[1]
    So = S["l2"][:, idx.order]

    qf = {}
    for e in EMBS:
        srt = -np.sort(-M[e], axis=1)
        qf[f"q_top1_{e}"] = srt[:, 0]
        qf[f"q_gap12_{e}"] = srt[:, 0] - srt[:, 1]
        qf[f"q_mean10_{e}"] = srt[:, :10].mean(1)
        qf[f"q_std10_{e}"] = srt[:, :10].std(1)
    qf["q_lognorm"] = ctx.lognorm[qr]
    qf["q_kp"] = ctx.kp[qr]
    qside, qvp = ctx.side[qr], ctx.vp[qr]
    qf["q_side"] = pd.Categorical(qside, categories=["L", "R", "N", "U"]).codes
    if transductive:
        Sqq = Z["l2"][0] @ Z["l2"][0].T
        np.fill_diagonal(Sqq, -np.inf)
        qf["q_nnq_cos"] = Sqq.max(1)
        qf["q_nnq_minus_top1"] = Sqq.max(1) - qf["q_top1_l2"]
        nnq = np.argsort(-Sqq, axis=1)[:, :5]
        qf["q_nnq_top1"] = qf["q_top1_l2"][nnq[:, 0]]

    rcfg = ctx.cfg["retrieval"]
    v25 = bool(rcfg.get("v25_stage1", False))
    lnb_k = int(rcfg.get("lnbnn_cand", 0))
    lnb_uk = int(rcfg.get("lnbnn_ucand", 0)) if v25 else 0
    Lu, Lim, ImgMat, UimgMat = fold_uimg_scores(fold, ctx, idx.ids) if rcfg.get("lnbnn", False) else (None, None, None, None)
    bv = _fold_v25(fold, ctx, idx, with_diff=True) if v25 else None
    compat_l2 = vp_compat_stats(Z["l2"][1], fold.gallery_labels, gvp) if v25 else None
    V25_EMBS = ("sw2152", "ssw2152", "sw_raw", "ssw_raw", "fl512", "fl512_raw", "cl2")
    V25_R, V25_Q = {}, {}
    if bv is not None:
        Mv = bv["M"]
        for e in V25_EMBS:
            V25_R[e] = np.argsort(np.argsort(-Mv[e], axis=1), axis=1)
            srt = -np.sort(-Mv[e], axis=1)
            V25_Q[f"q_{e}_max"] = srt[:, 0]
            V25_Q[f"q_{e}_gap"] = srt[:, 0] - srt[:, 1]
            V25_Q[f"q_{e}_mean10"] = srt[:, :10].mean(1)
            V25_Q[f"q_{e}_std10"] = srt[:, :10].std(1)
        for nm, mat in (("krec_sw10", bv["Mk_sw"]), ("diff_sw", bv["Md_sw"])):
            srt = -np.sort(-mat, axis=1)
            V25_Q[f"q_{nm}_max"] = srt[:, 0]
            V25_Q[f"q_{nm}_gap"] = srt[:, 0] - srt[:, 1]
            V25_Q[f"q_{nm}_mean10"] = srt[:, :10].mean(1)
            V25_Q[f"q_{nm}_std10"] = srt[:, :10].std(1)
        qf.update({k: v for k, v in V25_Q.items() if k.endswith(("_max", "_gap"))})

    KREC = {}
    if rcfg.get("krec", False):
        for e in ("l2", "lda"):
            Mk = idx.max(k_reciprocal_rerank(Z[e][0], Z[e][1], k1=20, k2=6, lam=0.3))
            srtk = -np.sort(-Mk, axis=1)
            KREC[e] = (Mk, srtk[:, 0], srtk[:, 0] - srtk[:, 1])
            qf[f"q_krec_{e}_max"] = srtk[:, 0]
            qf[f"q_krec_{e}_gap"] = srtk[:, 0] - srtk[:, 1]

    sift_on = ctx.pc is not None
    sift_top, per_id = ctx.cfg["sift"]["top_ids"], ctx.cfg["sift"]["imgs_per_id"]
    sw_k = int(rcfg.get("sw_cand", 15)) if v25 else 0
    fl_k = int(rcfg.get("fl_cand", 15)) if v25 else 0
    kr_k = int(rcfg.get("krec_sw_cand", 10)) if v25 else 0
    per_id_sw = int(rcfg.get("sift_imgs_sw", 2)) if v25 else 0
    k_im = int(rcfg.get("lnbnn_sift_imgs", 2))
    k_um = int(rcfg.get("lnbnn_uimg_sift_imgs", 1)) if v25 else 0
    ii = SF.index

    rows = []
    for i in range(nq):
        cands = list(np.argsort(comb[i])[:n_cand])
        cset_now = set(cands)

        def _add_top(scores, k):
            if scores is not None and k > 0:
                for c_extra in np.argsort(-scores)[:k]:
                    ci = int(c_extra)
                    if ci not in cset_now:
                        cands.append(ci)
                        cset_now.add(ci)

        _add_top(Lim[i] if Lim is not None else None, lnb_k)
        _add_top(Lu[i] if Lu is not None else None, lnb_uk)
        if bv is not None:
            _add_top(bv["M"]["sw2152"][i], sw_k)
            _add_top(bv["M"]["fl512"][i], fl_k)
            _add_top(bv["Mk_sw"][i], kr_k)
        cands = np.array(cands, dtype=int)
        r1 = set(np.argsort(-M["l2"][i])[:sift_top]) | set(np.argsort(-M["pcaw"][i])[:sift_top])
        if Lim is not None and lnb_k > 0:
            r1 |= set(int(x) for x in np.argsort(-Lim[i])[:lnb_k])
        if Lu is not None and lnb_uk > 0:
            r1 |= set(int(x) for x in np.argsort(-Lu[i])[:lnb_uk])
        if bv is not None:
            if sw_k > 0:
                r1 |= set(int(x) for x in np.argsort(-bv["M"]["sw2152"][i])[:sw_k])
            if fl_k > 0:
                r1 |= set(int(x) for x in np.argsort(-bv["M"]["fl512"][i])[:fl_k])
            if kr_k > 0:
                r1 |= set(int(x) for x in np.argsort(-bv["Mk_sw"][i])[:kr_k])

        for ci_pos, c in enumerate(cands):
            row = {"qi": i, "cand": c}
            for e in EMBS:
                row[f"max_{e}"] = M[e][i, c]
                row[f"top2_{e}"] = T2[e][i, c]
                row[f"proto_{e}"] = PR[e][i, c]
                row[f"rank_{e}"] = rank[e][i, c]
                row[f"gap_{e}"] = M[e][i, c] - qf[f"q_top1_{e}"][i]
                row[f"z_{e}"] = (M[e][i, c] - qf[f"q_mean10_{e}"][i]) / (qf[f"q_std10_{e}"][i] + 1e-6)
            for e, (Mk, m1, _) in KREC.items():
                row[f"krec_{e}"] = Mk[i, c]
                row[f"krec_{e}_gap"] = Mk[i, c] - m1[i]
            row["comb_rank"] = ci_pos
            if bv is not None:
                Mv = bv["M"]
                for e in V25_EMBS:
                    val = float(Mv[e][i, c])
                    row[f"max_{e}"] = val
                    row[f"rank_{e}"] = float(V25_R[e][i, c])
                    row[f"gap_{e}"] = val - float(V25_Q[f"q_{e}_max"][i])
                    row[f"z_{e}"] = (val - float(V25_Q[f"q_{e}_mean10"][i])) / (float(V25_Q[f"q_{e}_std10"][i]) + 1e-6)
                for nm, mat in (("krec_sw10", bv["Mk_sw"]), ("diff_sw", bv["Md_sw"])):
                    val = float(mat[i, c])
                    row[f"max_{nm}"] = val
                    row[f"gap_{nm}"] = val - float(V25_Q[f"q_{nm}_max"][i])
                    row[f"z_{nm}"] = (val - float(V25_Q[f"q_{nm}_mean10"][i])) / (float(V25_Q[f"q_{nm}_std10"][i]) + 1e-6)
            row["log_nimg"] = np.log(n_img[c])
            row["n_vp"] = n_vp[c]
            a, b = idx.starts[c], idx.counts[c]
            cols = idx.order[a:a + b]
            sims = So[i, a:a + b]
            jbest = cols[sims.argmax()]
            sd = qside[i]
            row["has_same_side"] = float(has_L[c] if sd == "L" else has_R[c] if sd == "R" else np.nan)
            same = gside[cols] == sd
            row["max_same_side"] = sims[same].max() if (sd in "LR" and same.any()) else np.nan
            opp = gside[cols] == ("R" if sd == "L" else "L")
            row["max_opp_side"] = sims[opp].max() if (sd in "LR" and opp.any()) else np.nan
            k = (qvp[i], gvp[jbest]) if qvp[i] <= gvp[jbest] else (gvp[jbest], qvp[i])
            row["compat_best"] = compat.get(k, np.nan)
            row["max_minus_compat"] = M["l2"][i, c] - row["compat_best"]
            if bv is not None:
                c_l2 = compat_l2.get(k)
                row["z_compat_l2"] = (M["l2"][i, c] - c_l2[0]) / c_l2[1] if c_l2 is not None else np.nan
                sims_sw = bv["S_sw_raw"][i, cols]
                jbest_sw = cols[sims_sw.argmax()]
                row["max_same_side_sw"] = sims_sw[same].max() if (sd in "LR" and same.any()) else np.nan
                row["max_opp_side_sw"] = sims_sw[opp].max() if (sd in "LR" and opp.any()) else np.nan
                k_sw = (qvp[i], gvp[jbest_sw]) if qvp[i] <= gvp[jbest_sw] else (gvp[jbest_sw], qvp[i])
                c_sw = bv["compat_sw"].get(k_sw)
                row["compat_best_sw"] = c_sw[0] if c_sw is not None else np.nan
                row["max_minus_compat_sw"] = bv["M"]["sw_raw"][i, c] - row["compat_best_sw"]
                row["z_compat_sw"] = (bv["M"]["sw_raw"][i, c] - c_sw[0]) / c_sw[1] if c_sw is not None else np.nan
            n_closer = npool - np.searchsorted(GS[jbest], np.float32(M["l2"][i, c]), side="right")
            row["recip_rank"] = np.log1p(max(n_closer - 1, 0))
            if sift_on and c in r1:
                best = cols[np.argsort(-S["l2"][i, cols])[:per_id]]
                if bv is not None and per_id_sw > 0:
                    best = np.union1d(best, cols[np.argsort(-bv["S_sw_raw"][i, cols])[:per_id_sw]])
                if ImgMat is not None and k_im > 0:
                    best = np.union1d(best, cols[np.argsort(-ImgMat[i, cols])[:k_im]])
                if UimgMat is not None and k_um > 0:
                    best = np.union1d(best, cols[np.argsort(-UimgMat[i, cols])[:k_um]])
                F = np.array([ctx.pc.get(qr[i], gr[j]) for j in best], dtype=np.float32)
                inl = np.nan_to_num(F[:, ii("inl_aff")], nan=-1)
                k2 = np.argsort(-inl)
                row["sift_checked"] = len(best)
                row["sift_inl_max"] = np.nanmax(F[:, ii("inl_aff")]) if np.isfinite(F[:, ii("inl_aff")]).any() else np.nan
                row["sift_inl_strict_max"] = np.nanmax(F[:, ii("inl_aff_strict")]) if np.isfinite(F[:, ii("inl_aff_strict")]).any() else np.nan
                row["sift_inlh_max"] = np.nanmax(F[:, ii("inl_h")]) if np.isfinite(F[:, ii("inl_h")]).any() else np.nan
                row["sift_inl_top2"] = np.nansum(np.sort(np.nan_to_num(F[:, ii("inl_aff")]))[::-1][:2])
                row["sift_n6"] = float((np.nan_to_num(F[:, ii("inl_aff")]) >= 6).sum())
                row["sift_mutual_max"] = np.nanmax(F[:, ii("n_mutual")]) if np.isfinite(F[:, ii("n_mutual")]).any() else np.nan
                fb = F[k2[0]]
                row["sift_ratio_best"] = fb[ii("inl_ratio")]
                row["sift_overkp_best"] = fb[ii("inl_over_kp")]
                row["sift_sim_best"] = fb[ii("inl_sim")]
                row["sift_spread_best"] = fb[ii("spread_min")]
                row["sift_rot_best"] = fb[ii("abs_rot")]
                row["sift_scale_best"] = fb[ii("log_scale")]
                row["sift_kpmin_best"] = fb[ii("kp_min")]
                if rcfg.get("bg_clean", False):
                    rot_v = np.nan_to_num(F[:, ii("abs_rot")], nan=99.0)
                    spr_v = np.nan_to_num(F[:, ii("spread_min")], nan=1.0)
                    is_bg = (rot_v < 0.50) & (spr_v < 0.25)
                    inl_c = np.where(is_bg, 0.0, np.nan_to_num(F[:, ii("inl_aff")], nan=-1.0))
                    row["sift_inl_clean"] = float(inl_c.max()) if (inl_c >= 0).any() else np.nan
                    row["sift_n6_clean"] = float((inl_c >= 6).sum())
                    row["sift_best_is_bg"] = float(is_bg[k2[0]])
                if bv is not None and UimgMat is not None:
                    rot_v = np.nan_to_num(F[:, ii("abs_rot")], nan=99.0)
                    spr_v = np.nan_to_num(F[:, ii("spread_min")], nan=1.0)
                    inl_v = np.nan_to_num(F[:, ii("inl_aff")], nan=-1.0)
                    im_g = ImgMat[i, best]
                    uim_g = UimgMat[i, best]
                    ush_g = uim_g / np.maximum(im_g, 1e-6)
                    row["sift_best_ushare"] = float(ush_g[k2[0]])
                    opp_g = gside[best] == ("R" if sd == "L" else "L")
                    row["sift_opp_side_inl"] = float(inl_v[opp_g].max()) if (sd in "LR" and opp_g.any() and (inl_v[opp_g] >= 0).any()) else np.nan
                    row["sift_n_vps_hi"] = float(len(set(gvp[best][inl_v >= 10])))
                    if rcfg.get("bg_clean_v2_img", False):
                        is_bg2_g = ((rot_v < 0.50) & (spr_v < 0.25)) | ((rot_v < 0.50) & (ush_g < 0.35) & (inl_v >= 6))
                        inl_c2 = np.where(is_bg2_g, 0.0, inl_v)
                        row["sift_inl_clean2"] = float(inl_c2.max()) if (inl_c2 >= 0).any() else np.nan
                        row["sift_n6_clean2"] = float((inl_c2 >= 6).sum())
                        row["sift_is_bg2"] = float(is_bg2_g[k2[0]])
            elif sift_on:
                row["sift_checked"] = 0
            rows.append(row)

    df = pd.DataFrame(rows)
    for k, v in qf.items():
        df[k] = np.asarray(v)[df.qi.values]
    if sift_on:
        g = df.groupby("qi")
        df["q_sift_inl_max"] = g.sift_inl_max.transform("max")
        df["q_sift_n6_cands"] = g.sift_n6.transform(lambda s: (s > 0).sum())
        df["sift_inl_minus_qmax"] = df.sift_inl_max - df.q_sift_inl_max
        df["sift_inl_rank"] = g.sift_inl_max.rank(ascending=False, method="min")
        if rcfg.get("bg_clean", False):
            df["q_sift_inl_clean_max"] = g.sift_inl_clean.transform("max")
            df["sift_inl_clean_minus_qmax"] = df.sift_inl_clean - df.q_sift_inl_clean_max
        if "sift_inl_clean2" in df.columns:
            df["q_sift_inl_clean2_max"] = g.sift_inl_clean2.transform("max")
            df["sift_inl_clean2_minus_qmax"] = df.sift_inl_clean2 - df.q_sift_inl_clean2_max
    if transductive:
        top1p = M["pcaw"].argmax(1)
        vote, nbbest = [], []
        for i, c in zip(df.qi.values, df.cand.values):
            nb = nnq[i]
            w = Sqq[i, nb]
            ok = w > 0.5
            vote.append(float(((top1p[nb] == c) & ok).sum()))
            nbbest.append(float(np.max(np.minimum(w, M["l2"][nb, c]))) if len(nb) else np.nan)
        df["tr_vote"] = vote
        df["tr_nb_score"] = nbbest
        df["tr_nb_minus_max"] = df.tr_nb_score - df.max_l2
    if rcfg.get("lnbnn", False):
        df = add_lnbnn_features(df, fold, ctx, idx.ids)
    df["query_id"] = fold.query_ids[df.qi.values]
    df["cand_id"] = idx.ids[df.cand.values]
    df["species_is_leo"] = int(fold.species == "leopard")
    return df


def _fold_hubness(fold, ctx, split_id):
    Zg_l2 = ctx.emb["l2"][fold.gallery_rows]
    Zq_l2 = ctx.emb["l2"][fold.query_rows]
    glab = fold.gallery_labels
    f_lda = fit_lda(Zg_l2, glab, gamma=0.3, d_out=256)
    Zg_ld, Zq_ld = f_lda(Zg_l2), f_lda(Zq_l2)
    rg_l2 = rg_diff(Zg_l2, glab, K=10)
    rg_ld = rg_diff(Zg_ld, glab, K=10)
    Hl2 = (Zq_l2 @ Zg_l2.T) - 0.50 * rg_l2[None, :]
    Hld = (Zq_ld @ Zg_ld.T) - 0.75 * rg_ld[None, :]
    names, inv = np.unique(glab, return_inverse=True)
    nid = len(names)
    members = [np.where(inv == j)[0] for j in range(nid)]
    nq = len(fold.query_ids)
    Il2 = np.full((nq, nid), -9.0, np.float32)
    Ild = np.full((nq, nid), -9.0, np.float32)
    Bl2 = np.zeros((nq, nid), np.float32)
    Bld = np.zeros((nq, nid), np.float32)
    ar = np.arange(nq)
    for j, cols in enumerate(members):
        sub2 = Hl2[:, cols]
        idx2 = sub2.argmax(1)
        Il2[:, j] = sub2[ar, idx2]
        Bl2[:, j] = rg_l2[cols[idx2]]
        subd = Hld[:, cols]
        idxd = subd.argmax(1)
        Ild[:, j] = subd[ar, idxd]
        Bld[:, j] = rg_ld[cols[idxd]]
    top2_l2 = -np.sort(-Il2, axis=1)[:, :2]
    top2_ld = -np.sort(-Ild, axis=1)[:, :2]
    id_pos = {n: j for j, n in enumerate(names)}
    q_info = {}
    for i, qid in enumerate(fold.query_ids):
        m1 = float(top2_l2[i, 0])
        m2 = float(top2_l2[i, 1]) if nid > 1 else m1
        l1 = float(top2_ld[i, 0])
        l2 = float(top2_ld[i, 1]) if nid > 1 else l1
        q_info[(split_id, qid)] = (id_pos, Il2[i], Ild[i], Bl2[i], Bld[i], m1, m1 - m2, l1, l1 - l2)
    return q_info


def attach_hubness_and_rimg(data, ctx, splits, extra_splits, cv, test, extra):
    t0 = time.time()
    all_folds = [(r, sp, e[sp][0]) for r, e in enumerate(splits) for sp in SPECIES]
    all_folds += [(-1, sp, test_fold(data, sp)) for sp in SPECIES]
    all_folds += [(r + 10, sp, e[sp][0]) for r, e in enumerate(extra_splits) for sp in SPECIES]

    q_hub, q_rimg = {}, {}
    for sid, sp, f in all_folds:
        q_hub.update(_fold_hubness(f, ctx, sid))
        cand_ids = np.unique(f.gallery_labels)
        _, _, _, _, Lr, _ = fold_all_scores(f, ctx, cand_ids)
        for i, qid in enumerate(f.query_ids):
            v = Lr[i]
            o = np.argsort(-v)
            m1 = float(v[o[0]])
            m2 = float(v[o[1]]) if len(v) > 1 else 0.0
            q_rimg[(sid, qid)] = (cand_ids, v, m1, m1 - m2)

    hub_cols = ["hub_l2", "hub_lda", "hub_l2_gap", "hub_lda_gap", "best_g_hub_l2", "best_g_hub_lda",
                "q_hub_l2_max", "q_hub_l2_gap", "q_hub_lda_max", "q_hub_lda_gap"]
    rimg_cols = ["lnb_rimg", "lnb_rimg_rel", "q_lnb_rimg_max", "q_lnb_rimg_gap"]

    def _apply(df):
        if df is None:
            return None
        df = df.copy()
        h_rows, r_rows = [], []
        for r, q, c in zip(df["split"].values, df["query_id"].values, df["cand_id"].values):
            id_pos, il2, ild, bl2, bld, m1, mg, l1, lg = q_hub[(int(r), q)]
            j = id_pos.get(c, -1)
            if j >= 0:
                v2, vd = float(il2[j]), float(ild[j])
                h_rows.append((v2, vd, m1 - v2, l1 - vd, float(bl2[j]), float(bld[j]), m1, mg, l1, lg))
            else:
                h_rows.append((-9.0, -9.0, 9.0, 9.0, 0.0, 0.0, m1, mg, l1, lg))
            names, v, rm1, rmg = q_rimg[(int(r), q)]
            rj = int(np.searchsorted(names, c))
            val = float(v[rj]) if (rj < len(names) and names[rj] == c) else 0.0
            r_rows.append((val, val / rm1 if rm1 > 0 else 0.0, rm1, rmg))
        df[hub_cols] = np.array(h_rows, dtype=np.float32)
        df[rimg_cols] = np.array(r_rows, dtype=np.float32)
        u_share = df["lnb_uimg"].values / np.maximum(df["lnb_img"].values, 1e-6)
        df["lnb_uimg_share"] = u_share.astype(np.float32)
        df["q_lnb_uimg_share"] = (df["q_lnb_uimg_max"].values / np.maximum(df["q_lnb_img_max"].values, 1e-6)).astype(np.float32)
        return df

    cv_out, test_out, extra_out = _apply(cv), _apply(test), _apply(extra)
    print(f"[features] attached hubness + lnb_rimg + uimg_share in {time.time() - t0:.1f}s")
    return cv_out, test_out, extra_out

In [ ]:
t_prep = time.time()
DATA = load_data()
build_sift_pack(DATA)
ensure_lnbnn_cache(DATA, Path(CFG_V25["lnbnn_dir"]))

EMB = {
    "l2": embed_unsupervised(DATA.X, DATA.meta, "l2"),
    "pcaw": embed_unsupervised(DATA.X, DATA.meta, "pcaw", d=512, eps=1.0),
}
KP_COUNTS = Sift().n_kp().astype(float)
SPLITS = build_splits(DATA, CFG_V25["cv"], EMB["l2"])
EXTRA_SPLITS = build_splits(DATA, CFG_V25["extra_cv"], EMB["l2"])
PAIR_CACHE = PairCache(CFG_V25["sift"])
CTX = Context(DATA, EMB, KP_COUNTS, CFG_V25, PAIR_CACHE)
print(f"[prepare] finished in {time.time() - t_prep:.1f}s | train={len(DATA.train)} test={len(DATA.test)}")


def build_all_tables():
    hdict = {
        "cv": CFG_V25["cv"],
        "extra_cv": CFG_V25["extra_cv"],
        "retrieval": CFG_V25["retrieval"],
        "sift": {k: v for k, v in CFG_V25["sift"].items() if k != "workers"},
        "v": 5,
    }
    key = hashlib.md5(json.dumps(hdict, sort_keys=True).encode()).hexdigest()[:10]
    tname = f"tables_v25_{key}.pkl"
    tpath = CACHE_DIR / "tables" / tname
    pre_tpath = _find_first(tname, is_dir=False)
    if tpath.exists():
        print(f"[tables] loaded cached {tpath}")
        return pickle.loads(tpath.read_bytes())
    if pre_tpath is not None and pre_tpath.exists():
        print(f"[tables] loaded precomputed {pre_tpath}")
        return pickle.loads(pre_tpath.read_bytes())

    t0 = time.time()
    folds = [(r, sp, fold, truth) for r, e in enumerate(SPLITS) for sp, (fold, truth) in e.items()]
    efolds = [(r + 10, sp, fold, truth) for r, e in enumerate(EXTRA_SPLITS) for sp, (fold, truth) in e.items()]
    tfolds = [(sp, test_fold(DATA, sp)) for sp in SPECIES]

    pairs = []
    for _, _, fold, _ in folds + efolds:
        pairs += sift_pairs_for_fold(fold, CTX)
    for _, fold in tfolds:
        pairs += sift_pairs_for_fold(fold, CTX)
    print(f"[sift] {len(set(pairs)):,} unique Stage-1 SIFT pairs requested ({time.time() - t0:.1f}s to list)")
    PAIR_CACHE.ensure(pairs)

    n_cand = CFG_V25["retrieval"]["n_cand"]
    cv_dfs, extra_dfs, test_dfs = [], [], []
    for split_id, sp, fold, truth in folds:
        df = build_table(fold, CTX, n_cand, transductive=True)
        td = truth.as_dict()
        df["truth"] = df.query_id.map(td)
        df["y"] = (df.cand_id == df.truth).astype(int)
        df["is_new"] = (df.truth == NEW).astype(int)
        df["group"] = DATA.meta.identity.values[fold.query_rows[df.qi.values]]
        df["split"] = split_id
        df["species"] = sp
        cv_dfs.append(df)
        print(f"  [table] primary split {split_id} {sp}: {len(fold.query_rows)} queries ({time.time() - t0:.0f}s)")

    for split_id, sp, fold, truth in efolds:
        df = build_table(fold, CTX, n_cand, transductive=True)
        td = truth.as_dict()
        df["truth"] = df.query_id.map(td)
        df["y"] = (df.cand_id == df.truth).astype(int)
        df["is_new"] = (df.truth == NEW).astype(int)
        df["group"] = DATA.meta.identity.values[fold.query_rows[df.qi.values]]
        df["split"] = split_id
        df["species"] = sp
        extra_dfs.append(df)
        print(f"  [table] extra split {split_id} {sp}: {len(fold.query_rows)} queries ({time.time() - t0:.0f}s)")

    for sp, fold in tfolds:
        df = build_table(fold, CTX, n_cand, transductive=True)
        df["split"] = -1
        df["species"] = sp
        test_dfs.append(df)

    cv_df = pd.concat(cv_dfs, ignore_index=True)
    extra_df = pd.concat(extra_dfs, ignore_index=True)
    test_df = pd.concat(test_dfs, ignore_index=True)
    cv_df, test_df, extra_df = attach_hubness_and_rimg(DATA, CTX, SPLITS, EXTRA_SPLITS, cv_df, test_df, extra_df)

    out = {"cv": cv_df, "test": test_df, "extra": extra_df}
    tpath.parent.mkdir(parents=True, exist_ok=True)
    tpath.write_bytes(pickle.dumps(out))
    print(f"[tables] built & cached in {time.time() - t0:.1f}s -> {tpath.name}")
    return out


SHARED_TABLES = build_all_tables()
CV_IN = SHARED_TABLES["cv"]
TEST_IN = SHARED_TABLES["test"]
EXTRA_IN = SHARED_TABLES["extra"]
print(f"[shared] cv={CV_IN.shape}, extra={EXTRA_IN.shape}, test={TEST_IN.shape}")

In [ ]:
NON_FEATURES = {"p", "p_lgb", "p_xgb", "p_cat", "p_lr", "p_mlp", "p_hgb",
                "qi", "cand", "query_id", "cand_id", "y", "split", "group", "species", "is_new", "truth"}
MONO_SIM_POS = ("max_l2", "max_pcaw", "max_lda", "hub_l2", "hub_lda", "lnb_rimg",
                "max_lda512", "max_slda512", "max_sw2152", "max_ssw2152",
                "max_sw_raw", "max_ssw_raw", "max_fl512", "max_fl512_raw", "max_cl2",
                "max_krec_sw10", "max_diff_sw")


def feature_cols(df, drop=()):
    return [c for c in df.columns if c not in NON_FEATURES and c not in drop and df[c].dtype != object]


def softmax_groups(z, gid, T):
    z = z / T
    mx = np.full(gid.max() + 1, -np.inf)
    np.maximum.at(mx, gid, z)
    e = np.exp(z - mx[gid])
    s = np.zeros(gid.max() + 1)
    np.add.at(s, gid, e)
    return e / s[gid]


def fit_temperature(z, gid, y):
    has = np.zeros(gid.max() + 1, bool)
    has[gid[y == 1]] = True
    m = has[gid]
    zz, gg, yy = z[m], np.unique(gid[m], return_inverse=True)[1], y[m]

    def nll(T):
        p = softmax_groups(zz, gg, T)
        return -np.log(np.clip(p[yy == 1], 1e-12, 1)).mean()

    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x


def shift_logit(p, beta):
    return expit(logit(np.clip(p, 1e-6, 1 - 1e-6)) + beta)


def rank_labels(cands, probs, p_new, k=5):
    labels = list(cands) + [NEW]
    p = np.r_[np.asarray(probs, float), p_new]
    order = np.argsort(-p, kind="stable")
    return [labels[i] for i in order[:k]]


def ap_at_5(truth, labels):
    for k, lab in enumerate(labels[:5], 1):
        if lab == truth:
            return 1.0 / k
    return 0.0


def diagnostics(truth: dict, preds: dict, p_new: dict = None, cand: dict = None) -> dict:
    qs = list(truth)
    is_new = np.array([truth[q] == NEW for q in qs])
    apv = np.array([ap_at_5(truth[q], preds[q]) for q in qs])
    top1 = np.array([preds[q][0] == truth[q] for q in qs])
    d = {
        "map5": float(apv.mean()),
        "n": len(qs),
        "frac_new": float(is_new.mean()),
        "map5_known": float(apv[~is_new].mean()) if (~is_new).any() else np.nan,
        "map5_new": float(apv[is_new].mean()) if is_new.any() else np.nan,
        "top1_acc": float(top1.mean()),
    }
    if p_new is not None and 0 < is_new.sum() < len(qs):
        d["new_auc"] = float(roc_auc_score(is_new, [p_new[q] for q in qs]))
    if cand is not None and (~is_new).any():
        d["cand_recall"] = float(np.mean([truth[q] in cand[q] for q in np.array(qs)[~is_new]]))
    return d


def query_table(df, pcol="p", rich_q=True):
    df = df.copy()
    df["_key"] = df.split.astype(str) + "|" + df.query_id
    g = df.groupby("_key", sort=False)
    first = g.head(1).set_index("_key")
    q = pd.DataFrame(index=first.index)
    srt = g[pcol].apply(lambda s: np.sort(s.values)[::-1])
    q["p1"] = srt.map(lambda a: a[0])
    q["p2"] = srt.map(lambda a: a[1] if len(a) > 1 else 0)
    q["psum"] = g[pcol].sum()
    q["lgap"] = logit(np.clip(q.p1, 1e-6, 1 - 1e-6)) - logit(np.clip(q.p2, 1e-6, 1 - 1e-6))
    q["n_p10"] = g[pcol].apply(lambda s: (s > 0.1).sum())
    if rich_q:
        q["p1_minus_p2"] = q["p1"] - q["p2"]
        q["p1_over_psum"] = q["p1"] / np.maximum(q["psum"], 1e-6)
        q["p_entropy"] = g[pcol].apply(lambda s: float(-np.sum((s.values / max(s.values.sum(), 1e-9)) * np.log(np.clip(s.values / max(s.values.sum(), 1e-9), 1e-9, 1.0)))))
        q["p3"] = srt.map(lambda a: a[2] if len(a) > 2 else 0)
    top = df.loc[g[pcol].idxmax()].set_index("_key")
    top_cols = (
        "max_same_side", "has_same_side", "compat_best", "max_minus_compat", "recip_rank", "log_nimg",
        "max_opp_side", "sift_inl_max", "z_pcaw", "z_l2", "z_lda",
        "lnb_img", "lnb_img_rel", "lnb_img_share", "lnb_k3", "lnb_log", "lnb_rel",
        "krec_l2", "krec_lda", "lnb_uimg", "lnb_uimg_rel", "sift_inl_clean", "sift_best_is_bg",
        "lnb_uimg_share", "sift_inl_clean2", "sift_is_bg2",
        "hub_l2", "hub_lda", "best_g_hub_l2", "best_g_hub_lda", "lnb_rimg", "lnb_rimg_rel",
        "max_sw2152", "z_sw2152", "max_sw_raw", "z_sw_raw", "max_ssw2152",
        "max_fl512", "z_fl512", "max_fl512_raw", "max_cl2",
        "max_krec_sw10", "z_krec_sw10", "max_diff_sw", "z_diff_sw",
        "max_same_side_sw", "z_compat_l2", "z_compat_sw",
        "sift_n_vps_hi", "sift_opp_side_inl", "sift_best_ushare",
    )
    if rich_q:
        top_cols = top_cols + ("sift_rot_best", "sift_spread_best", "sift_n6_clean")
    for c in top_cols:
        if c in top:
            q[f"top_{c}"] = top[c]
    for c in first.columns:
        if c.startswith("q_") or c in ("species_is_leo",):
            q[c] = first[c]
    for c in ("split", "query_id", "species", "is_new", "group", "truth"):
        if c in first:
            q[c] = first[c]
    return q.reset_index(drop=True)


def decide(df, qdf, pnew_col, beta, T, method="softmax"):
    df = df.copy()
    df["_key"] = df.split.astype(str) + "|" + df.query_id
    qkey = qdf.split.astype(str) + "|" + qdf.query_id
    gid = pd.factorize(df["_key"])[0]
    Tv = df.species.map(T).values if isinstance(T, dict) else T
    df["post"] = softmax_groups(logit(np.clip(df.p.values, 1e-6, 1 - 1e-6)), gid, Tv)
    qb = qdf.species.map(beta).values if isinstance(beta, dict) else np.full(len(qdf), beta)
    pn = pd.Series(shift_logit(qdf[pnew_col].values, qb), index=qkey)
    out = {}
    for key, g in df.groupby("_key", sort=False):
        pi = pn[key]
        out[key] = rank_labels(g.cand_id.values, g.post.values * (1 - pi), pi)
    return out


def evaluate(df, preds, qdf=None, pnew_col=None):
    res = {}
    for (r, sp), g in df.groupby(["split", "species"]):
        q = g.drop_duplicates("query_id")
        truth = dict(zip(q.query_id, q.truth))
        pr = {qid: preds[f"{r}|{qid}"] for qid in truth}
        pnew = None
        if qdf is not None:
            qq = qdf[(qdf.split == r) & (qdf.species == sp)]
            pnew = dict(zip(qq.query_id, qq[pnew_col]))
        cand = g.groupby("query_id").cand_id.apply(set).to_dict()
        res[(r, sp)] = diagnostics(truth, pr, pnew, cand)
    splits = sorted({r for r, _ in res})
    sps = [sp for sp in SPECIES if any(k[1] == sp for k in res)]
    wsum = sum(W_TEST[sp] for sp in sps)
    overall = np.array([sum(W_TEST[sp] / wsum * res[(r, sp)]["map5"] for sp in sps) for r in splits])
    summ = {"cv_mean": float(overall.mean()), "cv_std": float(overall.std(ddof=1)), "per_split": overall.tolist()}
    for sp in sps:
        for k in ("map5", "map5_known", "map5_new", "top1_acc", "new_auc", "cand_recall", "frac_new"):
            vals = [res[(r, sp)].get(k, np.nan) for r in splits]
            summ[f"{k}_{sp}"] = float(np.nanmean(vals))
    return summ, res


def _query_arrays(df, qdf, pnew_col, T):
    df = df.copy()
    df["_key"] = df.split.astype(str) + "|" + df.query_id
    gid = pd.factorize(df["_key"])[0]
    df["post"] = softmax_groups(logit(np.clip(df.p.values, 1e-6, 1 - 1e-6)), gid,
                                df.species.map(T).values if isinstance(T, dict) else T)
    df = df.sort_values(["_key", "post"], ascending=[True, False])
    df["r"] = df.groupby("_key").cumcount()
    top = df[df.r < 5].pivot(index="_key", columns="r", values="post").reindex(columns=range(5)).fillna(0)
    tr = df[df.y == 1].set_index("_key").r
    q = qdf.assign(_key=qdf.split.astype(str) + "|" + qdf.query_id).set_index("_key")
    top = top.reindex(q.index).fillna(0)
    return dict(top=top.values, trank=tr.reindex(q.index).fillna(99).values.astype(int), is_new=q.is_new.values,
                pnew=q[pnew_col].values, species=q.species.values, split=q.split.values)


def fast_ap(A, beta):
    pi = shift_logit(A["pnew"], beta)
    cand = A["top"] * (1 - pi)[:, None]
    knew = 1 + (cand > pi[:, None]).sum(1)
    tr = A["trank"]
    pos = tr + 1 + (knew <= tr + 1)
    ap_known = np.where(pos <= 5, 1.0 / pos, 0.0)
    ap_new = np.where(knew <= 5, 1.0 / knew, 0.0)
    return np.where(A["is_new"] == 1, ap_new, np.where(tr < 99, ap_known, 0.0))


def tune_beta(df, qdf, pnew_col, T, grid=np.round(np.arange(-3, 3.01, 0.05), 2)):
    A = _query_arrays(df, qdf, pnew_col, T)
    best = {}
    for sp in SPECIES:
        m = A["species"] == sp
        sub = {k: v[m] for k, v in A.items()}
        scores = [np.mean([fast_ap(sub, b)[sub["split"] == r].mean() for r in np.unique(sub["split"])]) for b in grid]
        best[sp] = (float(grid[int(np.argmax(scores))]), float(np.max(scores)))
    return best


def validate_and_write_submission(preds: dict, test_meta: pd.DataFrame, train_meta: pd.DataFrame, path: Path):
    sub = pd.DataFrame({"id": test_meta.id.values, "predictions": [" ".join(preds[i]) for i in test_meta.id]})
    assert list(sub.columns) == ["id", "predictions"]
    assert len(sub) == len(test_meta) == 2791
    assert set(sub.id) == set(test_meta.id) and not sub.id.duplicated().any()
    ids_by_sp = train_meta.groupby("species").identity.apply(set).to_dict()
    sp = test_meta.set_index("id").species
    for tid, pred in zip(sub.id, sub.predictions):
        labs = pred.split(" ")
        assert 1 <= len(labs) <= 5 and len(set(labs)) == len(labs), (tid, pred)
        for lab in labs:
            assert lab == NEW or lab in ids_by_sp[sp[tid]], (tid, lab)
    path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(path, index=False)
    return sub

In [ ]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.03, num_leaves=31, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    verbose=-1, num_threads=N_THREADS, seed=2026,
)


class LGBModel:
    def __init__(self, params=None, mono=True, max_rounds=3000):
        self.params = dict(LGB_PARAMS, **(params or {}))
        self.mono = mono
        self.max_rounds = max_rounds

    def fit(self, X, y, groups, feats):
        p = dict(self.params)
        if self.mono:
            sift_mono = ("sift_inl_clean2",) if "sift_inl_clean2" in feats else ("sift_inl_clean",)
            mono_set = set(MONO_SIM_POS) | set(sift_mono)
            p["monotone_constraints"] = [1 if f in mono_set else 0 for f in feats]
            p["monotone_constraints_method"] = "advanced"
        tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=0).split(X, y, groups))
        dtr = lgb.Dataset(X[tr], y[tr], feature_name=feats, free_raw_data=False)
        dva = lgb.Dataset(X[va], y[va], reference=dtr)
        self.model = lgb.train(p, dtr, self.max_rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        self.best_iter = self.model.best_iteration
        return self

    def predict(self, X):
        return self.model.predict(X, num_iteration=self.best_iter)


def oof_predict_lgb(df, feats, label, groups, n_splits=5, mono=True, seeds=None):
    X = df[feats].to_numpy(np.float32)
    y = df[label].to_numpy()
    seeds = list(seeds) if seeds else [None]
    oof = np.zeros(len(df), dtype=np.float64)
    iters = []
    for sd in seeds:
        p = {"seed": int(sd), "bagging_seed": int(sd), "feature_fraction_seed": int(sd)} if sd is not None else {}
        for tr, va in GroupKFold(n_splits).split(X, y, groups):
            m = LGBModel(params=p, mono=mono).fit(X[tr], y[tr], groups[tr], feats)
            oof[va] += m.predict(X[va]) / len(seeds)
            iters.append(m.best_iter)
    return oof, iters


# ---- Query-Query Keypoint Graph (kpgraph) ----
_SIFT_INST = None


def _rootsift_row(row):
    global _SIFT_INST
    if _SIFT_INST is None:
        _SIFT_INST = Sift()
    d = _SIFT_INST[row][1].astype(np.float32).reshape(-1, 128)
    return np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1e-7))


def query_graph(fold, m=5, nprobe=64, use_hnsw=True):
    rows = fold.query_rows
    X_list = [_rootsift_row(r) for r in rows]
    owner = np.repeat(np.arange(len(rows)), [len(x) for x in X_list]).astype(np.int32)
    X = np.concatenate(X_list).astype(np.float32)
    if HAS_FAISS and use_hnsw:
        index = faiss.IndexHNSWFlat(128, 32)
        index.hnsw.efConstruction = 64
        index.hnsw.efSearch = max(64, int(nprobe))
        index.add(X)
        d, nb = index.search(X, 30)
    elif HAS_FAISS:
        nlist = int(min(1024, max(64, len(X) // 1000)))
        index = faiss.IndexIVFFlat(faiss.IndexFlatL2(128), 128, nlist)
        index.train(X[np.random.RandomState(0).choice(len(X), min(len(X), 100000), replace=False)])
        index.add(X)
        index.nprobe = nprobe
        d, nb = index.search(X, 30)
    else:
        # Chunked exact fallback
        ds, nbs = [], []
        for b in range(0, len(X), 2048):
            db, ib = _numpy_knn_l2(X[b:b + 2048], X, 30)
            ds.append(db)
            nbs.append(ib)
        d, nb = np.concatenate(ds), np.concatenate(nbs)
    nb = np.where(nb >= 0, owner[np.maximum(nb, 0)], -1)
    d = np.sqrt(np.maximum(d, 0))
    ok = (nb >= 0) & (nb != owner[:, None])
    rank = np.cumsum(ok, 1)
    has_bg = rank[:, -1] >= 11
    bg = np.where(has_bg, d[np.arange(len(d)), np.argmax(rank >= 11, 1)], np.max(np.where(ok, d, 0), 1))
    r, c = np.nonzero(ok & (rank <= 10))
    key = owner[r].astype(np.int64) * len(rows) + nb[r, c]
    dk = r.astype(np.int64) * len(rows) + nb[r, c]
    _, first = np.unique(dk, return_index=True)
    key, val = key[first], np.maximum(bg[r[first]] - d[r[first], c[first]], 0)
    uk, inv = np.unique(key, return_inverse=True)
    tot = np.bincount(inv, weights=val)
    score = {(int(k // len(rows)), int(k % len(rows))): float(v) for k, v in zip(uk, tot)}
    top = {}
    for (i, j), v in score.items():
        top.setdefault(i, []).append((v, j))
    top = {i: [j for _, j in sorted(l, reverse=True)[:m]] for i, l in top.items()}
    return score, top


# ---- Transductive Propagation ----
KP_FEATS = ["kp_qq", "kp_qq_sym", "kp_qq_rank"]
SIFT_BG_FEATS = ["sift_rot", "sift_scale", "sift_is_bg", "sift_inl_clean"]
SIFT_BG_V2_FEATS = ["sift_abs_scale", "sift_is_zero_rot"]
SW_FEATS = ["cos_sw", "cos_fl512", "rank_sw_ij", "rank_sw_ji", "krec_qq_sw"]
PAIR_FEATS = [
    "cos_l2", "cos_pcaw", "rank_ij", "rank_ji", "same_side", "opp_side", "kp_min", "dlognorm",
    "sift_inl", "sift_inl_strict", "sift_mutual", "sift_ratio", "sift_overkp", "sift_spread",
    "top1_same", "p1_min", "species_is_leo",
]


def _krec_qq(Zq, Zg, k1=10, k2=4, lam=0.3):
    nq = len(Zq)
    Z = np.vstack([Zq, Zg]).astype(np.float32)
    N = len(Z)
    orig_dist = np.clip(2.0 - 2.0 * (Z @ Z.T), 0.0, 4.0)
    orig_dist = orig_dist / np.maximum(orig_dist.max(axis=1, keepdims=True), 1e-6)
    initial_rank = np.argsort(orig_dist, axis=1).astype(np.int32)
    V = np.zeros((N, N), dtype=np.float32)
    hk = int(round(k1 / 2)) + 1
    for i in range(N):
        fwd = initial_rank[i, :k1 + 1]
        bwd = initial_rank[fwd, :k1 + 1]
        k_recip = fwd[np.where(bwd == i)[0]]
        k_recip_exp = k_recip
        for cand in k_recip:
            cfwd = initial_rank[cand, :hk]
            cbwd = initial_rank[cfwd, :hk]
            c_recip = cfwd[np.where(cbwd == cand)[0]]
            if len(np.intersect1d(c_recip, k_recip)) > (2.0 / 3.0) * len(c_recip):
                k_recip_exp = np.append(k_recip_exp, c_recip)
        k_recip_exp = np.unique(k_recip_exp)
        w = np.exp(-orig_dist[i, k_recip_exp])
        V[i, k_recip_exp] = w / np.maximum(w.sum(), 1e-6)
    if k2 > 1:
        V = np.array([V[initial_rank[i, :k2]].mean(axis=0) for i in range(N)], dtype=np.float32)
    Vq = V[:nq]
    jaccard = np.ones((nq, nq), dtype=np.float32)
    for i in range(nq):
        nz = np.nonzero(Vq[i])[0]
        if len(nz):
            mn = np.minimum(Vq[i, nz][None, :], Vq[:, nz]).sum(axis=1)
            jaccard[i] = 1.0 - mn / (2.0 - mn)
    sim = 1.0 - ((1.0 - lam) * jaccard + lam * orig_dist[:nq, :nq])
    np.fill_diagonal(sim, -np.inf)
    return sim


def _sw_mats(fold, ctx):
    Zq_l2, Zg_l2 = ctx.emb["l2"][fold.query_rows], ctx.emb["l2"][fold.gallery_rows]
    projs = fit_sw_and_fullda(Zg_l2, fold.gallery_labels, reg=0.5, d_lda=512)
    Zq_sw, Zg_sw = projs["sw"](Zq_l2), projs["sw"](Zg_l2)
    Zq_fl = projs["fl512"](Zq_l2)
    S_sw = Zq_sw @ Zq_sw.T
    np.fill_diagonal(S_sw, -np.inf)
    S_fl = Zq_fl @ Zq_fl.T
    np.fill_diagonal(S_fl, -np.inf)
    order = np.argsort(-S_sw, axis=1)
    rank_sw = np.empty_like(order)
    rank_sw[np.arange(len(S_sw))[:, None], order] = np.arange(S_sw.shape[1])[None, :]
    Kr_sw = _krec_qq(Zq_sw, Zg_sw, k1=10, k2=4, lam=0.3)
    return S_sw, S_fl, rank_sw, Kr_sw, order[:, :5], np.argsort(-Kr_sw, axis=1)[:, :5]


def _posteriors(df, qdf, T, beta):
    df = df.copy()
    df["_key"] = df.split.astype(str) + "|" + df.query_id
    gid = pd.factorize(df["_key"])[0]
    df["post"] = softmax_groups(logit(np.clip(df.p.values, 1e-6, 1 - 1e-6)), gid, df.species.map(T).values)
    qkey = qdf.split.astype(str) + "|" + qdf.query_id
    pn = pd.Series(shift_logit(qdf.pnew.values, qdf.species.map(beta).values), index=qkey.values)
    return {k: (list(g.cand_id.values) + [NEW], np.r_[g.post.values * (1 - pn[k]), pn[k]]) for k, g in df.groupby("_key", sort=False)}


def _pair_table(fold, ctx, k, post, split, kpg, swm):
    Z = ctx.emb["l2"][fold.query_rows]
    Zp = ctx.emb["pcaw"][fold.query_rows]
    S = Z @ Z.T
    np.fill_diagonal(S, -np.inf)
    order = np.argsort(-S, axis=1)
    rank = np.empty_like(order)
    rank[np.arange(len(S))[:, None], order] = np.arange(S.shape[1])[None, :]
    nn = order[:, :k]
    rows = []
    qr, qids = fold.query_rows, fold.query_ids
    side, kp, ln = ctx.side, ctx.kp, ctx.lognorm
    kscore, ktop = kpg
    S_sw, S_fl, rank_sw, Kr_sw, top_sw, top_kr_sw = swm
    ii = SF.index
    for i in range(len(qr)):
        li, pi_ = post[f"{split}|{qids[i]}"]
        cand = list(nn[i]) + [j for j in ktop.get(i, []) if j not in set(nn[i])]
        cset = set(cand)
        for j in list(top_sw[i]) + list(top_kr_sw[i]):
            if int(j) not in cset:
                cand.append(int(j))
                cset.add(int(j))
        for j in cand:
            if j == i:
                continue
            lj, pj = post[f"{split}|{qids[j]}"]
            f = ctx.pc.get(qr[i], qr[j])
            if f is None:
                f = np.full(NF, np.nan, np.float32)
            si, sj = side[qr[i]], side[qr[j]]
            kp1, kp2 = kscore.get((i, j), 0.0), kscore.get((j, i), 0.0)
            rt = float(f[ii("abs_rot")])
            sc_ = float(f[ii("log_scale")])
            sp_ = float(f[ii("spread_min")])
            inl_ = float(f[ii("inl_aff")])
            isbg = float(np.isfinite(rt) and np.isfinite(sp_) and rt < 0.50 and sp_ < 0.25)
            rows.append((
                i, j, S[i, j], float(Zp[i] @ Zp[j]), rank[i, j], rank[j, i],
                float(si == sj and si in "LR"), float({si, sj} == {"L", "R"}),
                min(kp[qr[i]], kp[qr[j]]), abs(ln[qr[i]] - ln[qr[j]]),
                f[ii("inl_aff")], f[ii("inl_aff_strict")], f[ii("n_mutual")],
                f[ii("inl_ratio")], f[ii("inl_over_kp")], f[ii("spread_min")],
                float(li[int(np.argmax(pi_))] == lj[int(np.argmax(pj))]), float(min(pi_.max(), pj.max())),
                int(fold.species == "leopard"),
                kp1, kp2, ktop[i].index(j) if j in ktop.get(i, []) else 99,
                rt, sc_, isbg, 0.0 if isbg else inl_,
                abs(sc_) if np.isfinite(sc_) else np.nan, float(np.isfinite(rt) and rt < 0.15),
                float(S_sw[i, j]), float(S_fl[i, j]), int(rank_sw[i, j]), int(rank_sw[j, i]), float(Kr_sw[i, j]),
            ))
    cols = ["i", "j"] + PAIR_FEATS + KP_FEATS + SIFT_BG_FEATS + SIFT_BG_V2_FEATS + SW_FEATS
    return pd.DataFrame(rows, columns=cols)


def _propagate(n, labels_per_q, probs_per_q, edges, w, alpha, iters=30):
    if alpha == 0 or len(edges) == 0:
        return list(zip(labels_per_q, probs_per_q)), np.zeros(n, dtype=bool)
    lab_index = {}
    for labs in labels_per_q:
        for l in labs:
            lab_index.setdefault(l, len(lab_index))
    P0 = np.zeros((n, len(lab_index)))
    for q, (labs, pr) in enumerate(zip(labels_per_q, probs_per_q)):
        P0[q, [lab_index[l] for l in labs]] = pr
    i, j = edges[:, 0], edges[:, 1]
    W = sp.coo_matrix((np.r_[w, w], (np.r_[i, j], np.r_[j, i])), shape=(n, n)).tocsr()
    W.data = np.minimum(W.data, 1.0)
    d = np.asarray(W.sum(1)).ravel()
    dinv = np.where(d > 0, 1.0 / np.sqrt(np.maximum(d, 1e-12)), 0.0)
    Sn = sp.diags(dinv) @ W @ sp.diags(dinv)
    F = P0.copy()
    for _ in range(iters):
        F = alpha * (Sn @ F) + (1 - alpha) * P0
    F /= F.sum(1, keepdims=True)
    inv = np.array(list(lab_index.keys()), dtype=object)
    return [(list(inv[np.where(F[q] > 1e-9)[0]]), F[q, np.where(F[q] > 1e-9)[0]]) for q in range(n)], (d > 0)


_TRANS_SHARED_PREP = None


def _get_trans_shared(cfg):
    global _TRANS_SHARED_PREP
    if _TRANS_SHARED_PREP is not None:
        return _TRANS_SHARED_PREP
    tc = cfg["transductive"]
    k = cfg["sift"]["graph_k"]
    t0 = time.time()
    folds = [(r, sp_, e[sp_][0]) for r, e in enumerate(SPLITS) for sp_ in SPECIES]
    tfolds = [(-1, sp_, test_fold(DATA, sp_)) for sp_ in SPECIES]
    KP, SWM = {}, {}
    for r, sp_, f in folds + tfolds:
        h = hashlib.md5(np.asarray(f.query_rows).tobytes()).hexdigest()[:12]
        kpf = f"{sp_}_{h}_hnsw.pkl"
        cp = CACHE_DIR / "kpgraph" / kpf
        pre_cp = (PRE_KPGRAPH_DIR / kpf) if PRE_KPGRAPH_DIR is not None else None
        if cp.exists():
            KP[(r, sp_)] = pickle.loads(cp.read_bytes())
        elif pre_cp is not None and pre_cp.exists():
            KP[(r, sp_)] = pickle.loads(pre_cp.read_bytes())
        else:
            KP[(r, sp_)] = query_graph(f, nprobe=int(tc.get("kp_nprobe", 64)), use_hnsw=True)
            cp.parent.mkdir(parents=True, exist_ok=True)
            cp.write_bytes(pickle.dumps(KP[(r, sp_)]))
        SWM[(r, sp_)] = _sw_mats(f, CTX)
    pairs = []
    for r, sp_, f in folds + tfolds:
        pairs += query_neighbour_pairs(f, CTX, k)
        pairs += [(f.query_rows[i], f.query_rows[j]) for i, js in KP[(r, sp_)][1].items() for j in js]
        for i in range(len(SWM[(r, sp_)][4])):
            pairs += [(f.query_rows[i], f.query_rows[int(j)]) for j in SWM[(r, sp_)][4][i]]
            pairs += [(f.query_rows[i], f.query_rows[int(j)]) for j in SWM[(r, sp_)][5][i]]
    PAIR_CACHE.ensure(pairs)
    print(f"[trans_prep] kpgraph + sw2152 qq-affinity + query SIFT cache ready in {time.time() - t0:.1f}s")
    _TRANS_SHARED_PREP = (folds, tfolds, KP, SWM)
    return _TRANS_SHARED_PREP


def apply_transductive(cfg, cv, qdf, test, tq, T, beta):
    tc = cfg["transductive"]
    k = cfg["sift"]["graph_k"]
    t0 = time.time()
    folds, tfolds, KP, SWM = _get_trans_shared(cfg)
    PF = PAIR_FEATS + KP_FEATS + SIFT_BG_FEATS + SIFT_BG_V2_FEATS + SW_FEATS
    post_cv = _posteriors(cv, qdf, T, beta)
    post_te = _posteriors(test, tq, T, beta)
    ident = DATA.meta.identity.values
    ptabs = []
    for r, sp_, f in folds:
        pt = _pair_table(f, CTX, k, post_cv, r, KP[(r, sp_)], SWM[(r, sp_)])
        pt["y"] = (ident[f.query_rows[pt.i.values]] == ident[f.query_rows[pt.j.values]]).astype(int)
        pt["group"] = ident[f.query_rows[pt.i.values]]
        pt["split"] = r
        pt["species"] = sp_
        ptabs.append(pt)
    pcv = pd.concat(ptabs, ignore_index=True)
    pcv["p_same"], _ = oof_predict_lgb(pcv, PF, "y", pcv.group.values, mono=True)
    print(f"[trans] pair OOF AUC={roc_auc_score(pcv.y, pcv.p_same):.4f} (cos AUC={roc_auc_score(pcv.y, pcv.cos_l2):.4f}) in {time.time() - t0:.1f}s")

    tt = []
    for r, sp_, f in tfolds:
        pt = _pair_table(f, CTX, k, post_te, r, KP[(r, sp_)], SWM[(r, sp_)])
        pt["split"] = r
        pt["species"] = sp_
        tt.append(pt)
    pte = pd.concat(tt, ignore_index=True)
    pm = LGBModel(mono=True).fit(pcv[PF].to_numpy(np.float32), pcv["y"].to_numpy(), pcv.group.values, PF)
    pte["p_same"] = pm.predict(pte[PF].to_numpy(np.float32))

    def _ap_q(labels, probs, truth, new_scale):
        pr = np.array(probs, float).copy()
        labs = list(labels)
        if NEW in labs:
            pr[labs.index(NEW)] *= new_scale
        for kk, o in enumerate(np.argsort(-pr, kind="stable")[:5], 1):
            if labs[o] == truth:
                return 1.0 / kk
        return 0.0

    def _rank_q(labels, probs, new_scale):
        pr = np.array(probs, float).copy()
        labs = list(labels)
        if NEW in labs:
            pr[labs.index(NEW)] *= new_scale
        else:
            labs.append(NEW)
            pr = np.r_[pr, 0.0]
        return [labs[o] for o in np.argsort(-pr, kind="stable")[:5]]

    def run_fold(post, ptab, f, r, alpha, tau_e):
        keys = [f"{r}|{q}" for q in f.query_ids]
        labs = [post[kk][0] for kk in keys]
        prs = [post[kk][1] for kk in keys]
        m = ptab.p_same.values > tau_e
        outp, conn = _propagate(len(keys), labs, prs, ptab[["i", "j"]].values[m], ptab.p_same.values[m], alpha)
        return keys, outp, conn

    truth_by_key = dict(zip(qdf.split.astype(str) + "|" + qdf.query_id, qdf.truth))
    grid_alpha = tc.get("alphas", [0.0, 0.4, 0.6, 0.75, 0.85])
    grid_tau = tc.get("tau_e", [0.35, 0.45, 0.55, 0.65, 0.75])
    scales = np.exp(np.arange(-1.5, 1.51, 0.25))
    outer_picks, best = {}, {}
    for sp_ in SPECIES:
        res = []
        sp_splits = [r for r, s2, _ in folds if s2 == sp_]
        for alpha in grid_alpha:
            for tau_e in (grid_tau if alpha > 0 else [1.0]):
                aps = {s: [] for s in range(len(scales))}
                for r, s2, f in folds:
                    if s2 != sp_:
                        continue
                    keys, outp, conn = run_fold(post_cv, pcv[(pcv.split == r) & (pcv.species == sp_)], f, r, alpha, tau_e)
                    for si, sc in enumerate(scales):
                        aps[si].append(np.mean([_ap_q(l, p, truth_by_key[kk], sc if c else 1.0) for kk, (l, p), c in zip(keys, outp, conn)]))
                for si, sc in enumerate(scales):
                    res.append((alpha, tau_e, float(np.log(sc)), float(np.mean(aps[si])), aps[si]))
        df_g = pd.DataFrame(res, columns=["alpha", "tau_e", "log_new_scale", "mapv", "per_split"])
        off0 = df_g[df_g.alpha == 0]
        off = off0.loc[off0.mapv.idxmax()]
        ps_mat = np.array(df_g.per_split.tolist())
        nested_ps, picks = [], []
        for idx_r, r_id in enumerate(sp_splits):
            train_mean = np.delete(ps_mat, idx_r, axis=1).mean(1)
            br = df_g.iloc[int(np.argmax(train_mean))]
            nested_ps.append(float(ps_mat[int(np.argmax(train_mean)), idx_r]))
            pk = dict(alpha=float(br.alpha), tau_e=float(br.tau_e), log_new_scale=float(br.log_new_scale))
            outer_picks[(r_id, sp_)] = pk
            picks.append(pk)
        b_al = float(np.median([p["alpha"] for p in picks]))
        b_ta = float(np.median([p["tau_e"] for p in picks]))
        b_sc = float(np.median([p["log_new_scale"] for p in picks]))
        best[sp_] = dict(alpha=b_al, tau_e=b_ta, log_new_scale=b_sc, map=float(np.mean(nested_ps)), per_split=nested_ps, map_off=float(off.mapv), per_split_off=list(off.per_split))
        print(f"[trans] {sp_} NESTED: alpha={b_al:.2f} tau_e={b_ta:.2f} new_scale=e^{b_sc:.2f} -> nested_MAP={np.mean(nested_ps):.4f} (alpha=0: {off.mapv:.4f})")

    export_cv, export_probs, preds = {}, {}, {}
    for r, sp_, f in folds:
        bb = outer_picks[(r, sp_)]
        keys, outp, conn = run_fold(post_cv, pcv[(pcv.split == r) & (pcv.species == sp_)], f, r, bb["alpha"], bb["tau_e"])
        sc_b = np.exp(bb["log_new_scale"])
        for kk, (l, p), c in zip(keys, outp, conn):
            ll, pp = list(l), np.array(p, float).copy()
            if NEW in ll:
                pp[ll.index(NEW)] *= (sc_b if c else 1.0)
            export_cv.setdefault(r, {})[kk.split("|", 1)[1]] = dict(zip(ll, pp / pp.sum()))
    for r, sp_, f in tfolds:
        bb = best[sp_]
        keys, outp, conn = run_fold(post_te, pte[pte.species == sp_], f, r, bb["alpha"], bb["tau_e"])
        sc_b = np.exp(bb["log_new_scale"])
        print(f"[trans] test {sp_}: connected={int(conn.sum())}/{len(conn)} ({conn.mean():.3f})")
        for kk, (l, p), c in zip(keys, outp, conn):
            sc_i = sc_b if c else 1.0
            qid = kk.split("|", 1)[1]
            preds[qid] = _rank_q(l, p, sc_i)
            ll, pp = list(l), np.array(p, float).copy()
            if NEW in ll:
                pp[ll.index(NEW)] *= sc_i
            export_probs[qid] = dict(zip(ll, pp / pp.sum()))
    return preds, best, export_cv, export_probs

In [ ]:
class UnconstrainedLGB:
    def __init__(self, seed=2026, params=None):
        self.params = dict(
            objective="binary", learning_rate=0.025, num_leaves=47, min_child_samples=30,
            feature_fraction=0.65, bagging_fraction=0.80, bagging_freq=1, lambda_l2=3.0,
            verbose=-1, num_threads=N_THREADS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed,
        )
        if params:
            self.params.update(params)

    def fit(self, X, y, groups, feats):
        tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=self.params["seed"] % 1000).split(X, y, groups))
        dtr = lgb.Dataset(X[tr], y[tr], feature_name=feats, free_raw_data=False)
        dva = lgb.Dataset(X[va], y[va], reference=dtr)
        self.model = lgb.train(self.params, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        self.best_iter = self.model.best_iteration
        return self

    def predict(self, X):
        return np.clip(self.model.predict(X, num_iteration=self.best_iter), 1e-6, 1 - 1e-6)


class ExtraTreesLGB:
    def __init__(self, seed=2026):
        self.params = dict(
            objective="binary", extra_trees=True, learning_rate=0.03, num_leaves=63, min_child_samples=40,
            feature_fraction=0.50, bagging_fraction=0.75, bagging_freq=1, lambda_l2=5.0, lambda_l1=1.0,
            verbose=-1, num_threads=N_THREADS, seed=seed + 17, bagging_seed=seed + 17, feature_fraction_seed=seed + 17,
        )

    def fit(self, X, y, groups, feats):
        tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=(self.params["seed"] + 3) % 1000).split(X, y, groups))
        dtr = lgb.Dataset(X[tr], y[tr], feature_name=feats, free_raw_data=False)
        dva = lgb.Dataset(X[va], y[va], reference=dtr)
        self.model = lgb.train(self.params, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        self.best_iter = self.model.best_iteration
        return self

    def predict(self, X):
        return np.clip(self.model.predict(X, num_iteration=self.best_iter), 1e-6, 1 - 1e-6)


class LambdaRankLGB:
    def __init__(self, seed=2026):
        self.params = dict(
            objective="lambdarank", metric="ndcg", ndcg_eval_at=[1, 5], learning_rate=0.03,
            num_leaves=31, min_child_samples=25, feature_fraction=0.75, bagging_fraction=0.80,
            bagging_freq=1, lambda_l2=2.0, verbose=-1, num_threads=N_THREADS,
            seed=seed, bagging_seed=seed, feature_fraction_seed=seed,
        )

    @staticmethod
    def _gcounts(qkeys):
        _, idx, counts = np.unique(qkeys, return_index=True, return_counts=True)
        return counts[np.argsort(idx)]

    def fit(self, X, y, groups, feats, qkeys):
        tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=self.params["seed"] % 1000).split(X, y, groups))
        tr_ord = tr[np.argsort(qkeys[tr], kind="stable")]
        va_ord = va[np.argsort(qkeys[va], kind="stable")]
        dtr = lgb.Dataset(X[tr_ord], y[tr_ord], group=self._gcounts(qkeys[tr_ord]), feature_name=feats, free_raw_data=False)
        dva = lgb.Dataset(X[va_ord], y[va_ord], group=self._gcounts(qkeys[va_ord]), reference=dtr)
        self.model = lgb.train(self.params, dtr, 2500, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
        self.best_iter = self.model.best_iteration
        return self

    def predict_raw(self, X):
        return self.model.predict(X, num_iteration=self.best_iter)


class GPUXGBModel:
    def __init__(self, seed=2026):
        self.seed = seed

    def fit(self, X, y, groups, feats):
        import xgboost as xgb
        tr, va = next(GroupShuffleSplit(1, test_size=0.15, random_state=self.seed % 1000).split(X, y, groups))
        self.model = xgb.XGBClassifier(
            n_estimators=3000, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.7,
            min_child_weight=10, reg_lambda=3.0, tree_method="hist", device="cuda",
            early_stopping_rounds=100, eval_metric="logloss", random_state=self.seed,
        )
        self.model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        return self

    def predict(self, X):
        return np.clip(self.model.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)


class SGDLogModel:
    def __init__(self, seed=2026):
        self.seed = seed

    def fit(self, X, y, groups, feats):
        self.model = make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(),
            SGDClassifier(loss="log_loss", alpha=3e-4, max_iter=35, tol=1e-3, early_stopping=True, validation_fraction=0.1, random_state=self.seed),
        )
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return np.clip(self.model.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)


class LRCalibModel:
    def __init__(self, C=0.3):
        self.C = C

    def fit(self, X, y, groups, feats):
        self.model = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), LogisticRegression(C=self.C, max_iter=3000))
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return np.clip(self.model.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)


class MLPCalibModel:
    def __init__(self, seed=2026):
        self.seed = seed

    def fit(self, X, y, groups, feats):
        self.model = make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-3, early_stopping=True, validation_fraction=0.15, max_iter=200, random_state=self.seed),
        )
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return np.clip(self.model.predict_proba(X)[:, 1], 1e-6, 1 - 1e-6)


def oof_pnew_per_species(qdf, qfeats, seeds=(2026, 2027)):
    oof = np.zeros(len(qdf), dtype=np.float64)
    mods = {}
    for sp in SPECIES:
        idx = np.where((qdf.species == sp).values)[0]
        sub = qdf.iloc[idx].reset_index(drop=True)
        X = sub[qfeats].to_numpy(np.float32)
        y = sub["is_new"].to_numpy()
        grp = sub["group"].values
        p_lgb = np.zeros(len(sub), np.float64)
        p_mlp = np.zeros(len(sub), np.float64)
        p_lr = np.zeros(len(sub), np.float64)
        m_lgb, m_mlp, m_lr = [], [], []
        for si, sd in enumerate(seeds):
            for tr, va in GroupKFold(5).split(X, y, grp):
                ml = UnconstrainedLGB(seed=sd, params={"num_leaves": 23, "min_child_samples": 20, "lambda_l2": 2.0}).fit(X[tr], y[tr], grp[tr], qfeats)
                p_lgb[va] += ml.predict(X[va]) / len(seeds)
                m_lgb.append(ml)
                if si == 0:
                    mm = MLPCalibModel(seed=sd).fit(X[tr], y[tr], grp[tr], qfeats)
                    mr = LRCalibModel(C=0.3).fit(X[tr], y[tr], grp[tr], qfeats)
                    p_mlp[va] = mm.predict(X[va])
                    p_lr[va] = mr.predict(X[va])
                    m_mlp.append(mm)
                    m_lr.append(mr)
        oof[idx] = np.clip(0.65 * p_lgb + 0.20 * p_mlp + 0.15 * p_lr, 1e-6, 1 - 1e-6)
        mods[sp] = (m_lgb, m_mlp, m_lr)
    return oof, mods


def predict_pnew_per_species(mods, tq, qfeats):
    out = np.zeros(len(tq), dtype=np.float64)
    for sp in SPECIES:
        idx = np.where((tq.species == sp).values)[0]
        if not len(idx):
            continue
        X = tq.iloc[idx][qfeats].to_numpy(np.float32)
        m_lgb, m_mlp, m_lr = mods[sp]
        pl = np.mean([m.predict(X) for m in m_lgb], axis=0)
        pm = np.mean([m.predict(X) for m in m_mlp], axis=0)
        pr = np.mean([m.predict(X) for m in m_lr], axis=0)
        out[idx] = np.clip(0.65 * pl + 0.20 * pm + 0.15 * pr, 1e-6, 1 - 1e-6)
    return out


# ---------------- Paradigm 1: v25_hnsw_rescue ----------------
def run_v25_paradigm1(cfg, cv_in, test_in, extra_in):
    t0 = time.time()
    common = [c for c in cv_in.columns if c in extra_in.columns]
    cv_all = pd.concat([cv_in[common], extra_in[common]], ignore_index=True)
    test = test_in.copy()
    feats = feature_cols(cv_all)
    groups = cv_all.group.values

    cv_all["p"], iters = oof_predict_lgb(cv_all, feats, "y", groups, mono=True)
    print(f"[v25 P1] {len(feats)} feats OOF done in {time.time() - t0:.1f}s, iters={iters}")
    qdf_all = query_table(cv_all, rich_q=True)
    qfeats = feature_cols(qdf_all)
    qdf_all["pnew"], pnew_sp_mods = oof_pnew_per_species(qdf_all, qfeats, seeds=(2026, 2027))

    cv = cv_all[cv_all.split < 10].reset_index(drop=True)
    qdf = qdf_all[qdf_all.split < 10].reset_index(drop=True)
    T = {sp: float(fit_temperature(logit(np.clip(cv[cv.species == sp].p.values, 1e-6, 1 - 1e-6)),
                                   pd.factorize(cv[cv.species == sp].split.astype(str) + "|" + cv[cv.species == sp].query_id)[0],
                                   cv[cv.species == sp].y.values)) for sp in SPECIES}
    beta = {sp: v[0] for sp, v in tune_beta(cv, qdf, "pnew", T).items()}
    summ, _ = evaluate(cv, decide(cv, qdf, "pnew", beta, T), qdf, "pnew")
    print(f"[v25 P1 Stage-2] CV={summ['cv_mean']:.5f} ± {summ['cv_std']:.5f} (h={summ['map5_hyena']:.5f}, l={summ['map5_leopard']:.5f}) | beta={beta}")

    rer = LGBModel(mono=True).fit(cv_all[feats].to_numpy(np.float32), cv_all["y"].to_numpy(), groups, feats)
    test["p"] = rer.predict(test[feats].to_numpy(np.float32))
    tq = query_table(test, rich_q=True)
    tq["pnew"] = predict_pnew_per_species(pnew_sp_mods, tq, qfeats)

    preds, best_t, exp_cv, exp_probs = apply_transductive(cfg, cv, qdf, test, tq, T, beta)
    ps = np.array([sum(W_TEST[sp] * best_t[sp]["per_split"][r] for sp in SPECIES) for r in range(len(SPLITS))])
    print(f"[v25 P1 Transductive] CV={ps.mean():.5f} ± {ps.std(ddof=1):.5f} | total {time.time() - t0:.1f}s")
    return preds, exp_cv, exp_probs, ps


# ---------------- Paradigm 2: v26_dual_r2 ----------------
def _oof_ensemble(df, feats, label, groups, qkeys_col, seeds=(2026,), use_gpu_xgb=False):
    X = np.nan_to_num(df[feats].to_numpy(np.float32), nan=np.nan, posinf=1e4, neginf=-1e4)
    y = df[label].to_numpy()
    qk = df[qkeys_col].values
    ulgb_oof = np.zeros(len(df), np.float64)
    et_oof = np.zeros(len(df), np.float64)
    rank_oof_raw = np.zeros(len(df), np.float64)
    sgd_oof = np.zeros(len(df), np.float64)
    xgb_oof = np.zeros(len(df), np.float64)
    ulgb_mods, et_mods, rank_mods, sgd_mods, xgb_mods = [], [], [], [], []
    for sd in seeds:
        for tr, va in GroupKFold(5).split(X, y, groups):
            m1 = UnconstrainedLGB(seed=sd).fit(X[tr], y[tr], groups[tr], feats)
            ulgb_oof[va] += m1.predict(X[va]) / len(seeds)
            ulgb_mods.append(m1)
            m2 = ExtraTreesLGB(seed=sd).fit(X[tr], y[tr], groups[tr], feats)
            et_oof[va] += m2.predict(X[va]) / len(seeds)
            et_mods.append(m2)
            if sd == seeds[0]:
                m4 = SGDLogModel(seed=sd).fit(X[tr], y[tr], groups[tr], feats)
                sgd_oof[va] = m4.predict(X[va])
                sgd_mods.append(m4)
            m3 = LambdaRankLGB(seed=sd).fit(X[tr], y[tr], groups[tr], feats, qk[tr])
            rank_oof_raw[va] += m3.predict_raw(X[va]) / len(seeds)
            rank_mods.append(m3)
            if use_gpu_xgb:
                m5 = GPUXGBModel(seed=sd).fit(X[tr], y[tr], groups[tr], feats)
                xgb_oof[va] += m5.predict(X[va]) / len(seeds)
                xgb_mods.append(m5)
    out = {
        "ulgb": np.clip(ulgb_oof, 1e-6, 1 - 1e-6),
        "etlgb": np.clip(et_oof, 1e-6, 1 - 1e-6),
        "sgd": np.clip(sgd_oof, 1e-6, 1 - 1e-6),
        "rank_raw": rank_oof_raw,
        "rank_p": np.clip(softmax_groups(rank_oof_raw, pd.factorize(qk)[0], 1.0), 1e-6, 1 - 1e-6),
    }
    r_cent = rank_oof_raw - pd.Series(rank_oof_raw).groupby(qk).transform("mean").values
    if use_gpu_xgb:
        out["xgb"] = np.clip(xgb_oof, 1e-6, 1 - 1e-6)
        l_mix = 0.30 * logit(out["ulgb"]) + 0.20 * logit(out["etlgb"]) + 0.25 * r_cent + 0.15 * logit(out["xgb"]) + 0.10 * logit(out["sgd"])
    else:
        l_mix = 0.35 * logit(out["ulgb"]) + 0.25 * logit(out["etlgb"]) + 0.30 * r_cent + 0.10 * logit(out["sgd"])
    out["mix"] = expit(l_mix)
    return out, {"ulgb": ulgb_mods, "etlgb": et_mods, "rank": rank_mods, "sgd": sgd_mods, "xgb": xgb_mods}


def _predict_ensemble(mods, X, qk, use_gpu_xgb=False):
    X = np.nan_to_num(X, nan=np.nan, posinf=1e4, neginf=-1e4)
    pu = np.clip(np.mean([m.predict(X) for m in mods["ulgb"]], axis=0), 1e-6, 1 - 1e-6)
    pe = np.clip(np.mean([m.predict(X) for m in mods["etlgb"]], axis=0), 1e-6, 1 - 1e-6)
    ps = np.clip(np.mean([m.predict(X) for m in mods["sgd"]], axis=0), 1e-6, 1 - 1e-6)
    rr = np.mean([m.predict_raw(X) for m in mods["rank"]], axis=0)
    r_cent = rr - pd.Series(rr).groupby(qk).transform("mean").values
    if use_gpu_xgb and mods["xgb"]:
        px = np.clip(np.mean([m.predict(X) for m in mods["xgb"]], axis=0), 1e-6, 1 - 1e-6)
        l_mix = 0.30 * logit(pu) + 0.20 * logit(pe) + 0.25 * r_cent + 0.15 * logit(px) + 0.10 * logit(ps)
    else:
        l_mix = 0.35 * logit(pu) + 0.25 * logit(pe) + 0.30 * r_cent + 0.10 * logit(ps)
    return {
        "ulgb": pu, "etlgb": pe, "sgd": ps,
        "rank_p": np.clip(softmax_groups(rr, pd.factorize(qk)[0], 1.0), 1e-6, 1 - 1e-6),
        "mix": expit(l_mix),
    }


def _attach_r2_neighbors(df, qdf, pcol, pnew_col, folds_by_split, k_nb=8):
    df = df.reset_index(drop=True).copy()
    df["_key"] = df.split.astype(str) + "|" + df.query_id
    post_map = {k: dict(zip(g.cand_id.values, g[pcol].values)) for k, g in df.groupby("_key", sort=False)}
    pnew_map = dict(zip((qdf.split.astype(str) + "|" + qdf.query_id).values, qdf[pnew_col].values))
    rk = df.groupby("_key")[pcol].rank(ascending=False, method="min").values.astype(np.float32)
    qmax = df.groupby("_key")[pcol].transform("max").values
    qtop2 = df.groupby("_key")[pcol].transform(lambda s: np.sort(s.values)[-2] if len(s) > 1 else s.values[0]).values
    df["r1_p"] = df[pcol].astype(np.float32)
    df["r1_rank"] = rk
    df["r1_gap_top"] = (df[pcol].values - np.where(df[pcol].values >= qmax - 1e-9, qtop2, qmax)).astype(np.float32)
    df["r1_pnew"] = df["_key"].map(pnew_map).astype(np.float32)

    nb_feats = np.zeros((len(df), 6), dtype=np.float32)
    ii_inl, ii_rot, ii_spr = SF.index("inl_aff"), SF.index("abs_rot"), SF.index("spread_min")
    for (r, sp), g in df.groupby(["split", "species"], sort=False):
        fold = folds_by_split[(int(r), sp)]
        qr, qids = fold.query_rows, fold.query_ids
        Zq_l2 = CTX.emb["l2"][qr]
        projs = fit_sw_and_fullda(CTX.emb["l2"][fold.gallery_rows], fold.gallery_labels, reg=0.5, d_lda=256)
        Zq_sw = projs["sw"](Zq_l2)
        S_sw = Zq_sw @ Zq_sw.T
        np.fill_diagonal(S_sw, -np.inf)
        S_l2 = Zq_l2 @ Zq_l2.T
        np.fill_diagonal(S_l2, -np.inf)
        comb = 0.65 * S_sw + 0.35 * S_l2
        nn = np.argsort(-comb, axis=1)[:, :k_nb]
        q_pos = {qid: i for i, qid in enumerate(qids)}
        nb_pnew_w = np.zeros(len(qr), dtype=np.float32)
        nb_cache = []
        for i in range(len(qr)):
            jlist = nn[i]
            w_raw = np.maximum(comb[i, jlist], 0.0)
            ok = (S_sw[i, jlist] > 0.25) | (S_l2[i, jlist] > 0.45)
            w_vec = np.where(ok, w_raw, 0.0)
            wsum = float(w_vec.sum())
            items, pnew_acc = [], 0.0
            for j, w in zip(jlist, w_vec):
                if w <= 0:
                    continue
                k_j = f"{int(r)}|{qids[j]}"
                d_j = post_map.get(k_j, {})
                pnew_acc += w * float(pnew_map.get(k_j, 0.5))
                top1_j = max(d_j, key=d_j.get) if d_j else None
                inl_c = 0.0
                f = PAIR_CACHE.get(qr[i], qr[j])
                if f is not None and np.isfinite(f[ii_inl]):
                    is_bg = np.isfinite(f[ii_rot]) and np.isfinite(f[ii_spr]) and f[ii_rot] < 0.50 and f[ii_spr] < 0.25
                    inl_c = 0.0 if is_bg else float(f[ii_inl])
                items.append((d_j, float(w), float(S_sw[i, j]), top1_j, inl_c))
            nb_pnew_w[i] = (pnew_acc / wsum) if wsum > 0 else float(pnew_map.get(f"{int(r)}|{qids[i]}", 0.5))
            nb_cache.append((items, wsum))
        for row_idx, qid, cid in zip(g.index.values, g.query_id.values, g.cand_id.values):
            qi = q_pos[qid]
            items, wsum = nb_cache[qi]
            if wsum <= 0:
                nb_feats[row_idx, 5] = nb_pnew_w[qi]
                continue
            pmax, pacc, tvote, ssupp, cmax = 0.0, 0.0, 0.0, 0.0, 0.0
            for d_j, w, csw, top1_j, inl_c in items:
                pj = float(d_j.get(cid, 0.0))
                pmax = max(pmax, min(w, pj))
                pacc += w * pj
                if top1_j == cid:
                    tvote += w
                if pj > 0.05 and csw > cmax:
                    cmax = csw
                ssupp = max(ssupp, inl_c * pj)
            nb_feats[row_idx] = (pmax, pacc / wsum, tvote / wsum, ssupp, cmax, nb_pnew_w[qi])
    df[["nb_p1_max", "nb_p1_wmean", "nb_top1_votes", "nb_sift_support", "nb_cos_sw_best", "q_nb_pnew_wmean"]] = nb_feats
    df.drop(columns=["_key"], inplace=True)
    return df


def run_v26_paradigm2(cfg, cv_in, test_in, extra_in):
    t0 = time.time()
    mcfg = cfg["model"]
    seeds = mcfg.get("bag_seeds", [2026])
    use_gpu_xgb = bool(mcfg.get("use_gpu_xgb", False))
    folds_by_split = {(r, sp): e[sp][0] for r, e in enumerate(SPLITS) for sp in SPECIES}
    for sp in SPECIES:
        folds_by_split[(-1, sp)] = test_fold(DATA, sp)
    for r, e in enumerate(EXTRA_SPLITS):
        for sp in SPECIES:
            folds_by_split[(r + 10, sp)] = e[sp][0]

    common = [c for c in cv_in.columns if c in extra_in.columns]
    cv = pd.concat([cv_in[common], extra_in[common]], ignore_index=True)
    test = test_in.copy()
    cv["_qkey"] = cv.split.astype(str) + "|" + cv.query_id
    test["_qkey"] = test.split.astype(str) + "|" + test.query_id

    feats_r1 = [c for c in feature_cols(cv) if c != "_qkey"]
    r1_oof, r1_mods = _oof_ensemble(cv, feats_r1, "y", cv.group.values, "_qkey", seeds=seeds, use_gpu_xgb=use_gpu_xgb)
    cv["p_r1"], cv["p_r1_rank"], cv["p_r1_ulgb"], cv["p_r1_etlgb"] = r1_oof["mix"], r1_oof["rank_p"], r1_oof["ulgb"], r1_oof["etlgb"]
    print(f"[v26 R1] {len(feats_r1)} feats OOF done in {time.time() - t0:.1f}s")
    qdf_r1 = query_table(cv, pcol="p_r1", rich_q=True)
    qfeats_r1 = [c for c in feature_cols(qdf_r1) if c != "_qkey"]
    pn1_oof, pn1_mods = oof_pnew_per_species(qdf_r1, qfeats_r1, seeds=(2026, 2027))
    qdf_r1["pnew_r1"] = pn1_oof

    r1_te = _predict_ensemble(r1_mods, test[feats_r1].to_numpy(np.float32), test["_qkey"].values, use_gpu_xgb=use_gpu_xgb)
    test["p_r1"], test["p_r1_rank"], test["p_r1_ulgb"], test["p_r1_etlgb"] = r1_te["mix"], r1_te["rank_p"], r1_te["ulgb"], r1_te["etlgb"]
    tq_r1 = query_table(test, pcol="p_r1", rich_q=True)
    tq_r1["pnew_r1"] = predict_pnew_per_species(pn1_mods, tq_r1, qfeats_r1)

    t1 = time.time()
    cv = _attach_r2_neighbors(cv, qdf_r1, "p_r1", "pnew_r1", folds_by_split, k_nb=int(mcfg.get("r2_k", 8)))
    test = _attach_r2_neighbors(test, tq_r1, "p_r1", "pnew_r1", folds_by_split, k_nb=int(mcfg.get("r2_k", 8)))
    print(f"[v26 R2] neighbor stacking attached in {time.time() - t1:.1f}s")

    feats_r2 = [c for c in feature_cols(cv) if c != "_qkey"]
    r2_oof, r2_mods = _oof_ensemble(cv, feats_r2, "y", cv.group.values, "_qkey", seeds=seeds, use_gpu_xgb=use_gpu_xgb)
    cv["p"], cv["p_ulgb"], cv["p_rank"], cv["p_etlgb"] = r2_oof["mix"], r2_oof["ulgb"], r2_oof["rank_p"], r2_oof["etlgb"]
    qdf = query_table(cv, pcol="p", rich_q=True)
    q_extra_cols = ["q_nb_pnew_wmean", "r1_pnew", "nb_p1_max", "nb_p1_wmean", "nb_top1_votes", "nb_sift_support"]
    top_r2 = cv.loc[cv.groupby("_qkey")["p"].idxmax()].set_index("_qkey")
    q_idx = qdf.split.astype(str) + "|" + qdf.query_id
    for c in q_extra_cols:
        qdf[f"top_{c}"] = q_idx.map(top_r2[c]).values
    qfeats = [c for c in feature_cols(qdf) if c != "_qkey"]
    qdf["pnew"], pn2_mods = oof_pnew_per_species(qdf, qfeats, seeds=(2026, 2027))

    r2_te = _predict_ensemble(r2_mods, test[feats_r2].to_numpy(np.float32), test["_qkey"].values, use_gpu_xgb=use_gpu_xgb)
    test["p"], test["p_ulgb"], test["p_rank"], test["p_etlgb"] = r2_te["mix"], r2_te["ulgb"], r2_te["rank_p"], r2_te["etlgb"]
    tq = query_table(test, pcol="p", rich_q=True)
    top_t2 = test.loc[test.groupby("_qkey")["p"].idxmax()].set_index("_qkey")
    tq_idx = tq.split.astype(str) + "|" + tq.query_id
    for c in q_extra_cols:
        tq[f"top_{c}"] = tq_idx.map(top_t2[c]).values
    tq["pnew"] = predict_pnew_per_species(pn2_mods, tq, qfeats)

    cv.drop(columns=["_qkey"], inplace=True)
    test.drop(columns=["_qkey"], inplace=True)
    cv = cv[cv.split < 10].reset_index(drop=True)
    qdf = qdf[qdf.split < 10].reset_index(drop=True)
    T = {sp: float(fit_temperature(logit(np.clip(cv[cv.species == sp].p.values, 1e-6, 1 - 1e-6)),
                                   pd.factorize(cv[cv.species == sp].split.astype(str) + "|" + cv[cv.species == sp].query_id)[0],
                                   cv[cv.species == sp].y.values)) for sp in SPECIES}
    beta = {sp: v[0] for sp, v in tune_beta(cv, qdf, "pnew", T).items()}
    summ, _ = evaluate(cv, decide(cv, qdf, "pnew", beta, T), qdf, "pnew")
    print(f"[v26 R2 Stage-2] CV={summ['cv_mean']:.5f} ± {summ['cv_std']:.5f} (h={summ['map5_hyena']:.5f}, l={summ['map5_leopard']:.5f}) | beta={beta}")

    preds, best_t, exp_cv, exp_probs = apply_transductive(cfg, cv, qdf, test, tq, T, beta)
    ps = np.array([sum(W_TEST[sp] * best_t[sp]["per_split"][r] for sp in SPECIES) for r in range(len(SPLITS))])
    print(f"[v26 R2 Transductive] CV={ps.mean():.5f} ± {ps.std(ddof=1):.5f} | total {time.time() - t0:.1f}s")
    return preds, exp_cv, exp_probs, ps

In [ ]:
def _try_load_probs(name, work_te, work_cv, pre_te, pre_cv):
    if FORCE_RERUN:
        return None
    for te_p, cv_p in ((work_te, work_cv), (pre_te, pre_cv)):
        if te_p is not None and cv_p is not None and te_p.exists() and cv_p.exists():
            tp = pickle.loads(te_p.read_bytes())
            cp = pickle.loads(cv_p.read_bytes())
            if set(tp) == set(DATA.test.id) and len(cp) == len(SPLITS):
                print(f"[resume] loaded cached {name} probabilities from {te_p.parent}")
                return cp, tp
    return None


# 1. Run or resume Paradigm 1 (v25_hnsw_rescue) and Paradigm 2 (v26_dual_r2)
res_v25 = _try_load_probs("v25", WORK_DIR / "v25.probs.pkl", WORK_DIR / "v25.cv.probs.pkl", PRE_V25_PROBS, PRE_V25_CV_PROBS)
if res_v25 is not None:
    cv_probs_v25, test_probs_v25 = res_v25
else:
    preds_v25, cv_probs_v25, test_probs_v25, ps_v25 = run_v25_paradigm1(CFG_V25, CV_IN, TEST_IN, EXTRA_IN)
    validate_and_write_submission(preds_v25, DATA.test, DATA.train, WORK_DIR / "submission_v25.csv")
    pickle.dump(test_probs_v25, open(WORK_DIR / "v25.probs.pkl", "wb"))
    pickle.dump(cv_probs_v25, open(WORK_DIR / "v25.cv.probs.pkl", "wb"))

res_v26 = _try_load_probs("v26", WORK_DIR / "v26.probs.pkl", WORK_DIR / "v26.cv.probs.pkl", PRE_V26_PROBS, PRE_V26_CV_PROBS)
if res_v26 is not None:
    cv_probs_v26, test_probs_v26 = res_v26
else:
    preds_v26, cv_probs_v26, test_probs_v26, ps_v26 = run_v26_paradigm2(CFG_V26, CV_IN, TEST_IN, EXTRA_IN)
    validate_and_write_submission(preds_v26, DATA.test, DATA.train, WORK_DIR / "submission_v26.csv")
    pickle.dump(test_probs_v26, open(WORK_DIR / "v26.probs.pkl", "wb"))
    pickle.dump(cv_probs_v26, open(WORK_DIR / "v26.cv.probs.pkl", "wb"))

# 2. Honest Nested Leave-One-Split-Out Blend Evaluation (matches scripts/eval_blend.py)
truths = {(r, sp): SPLITS[r][sp][1].as_dict() for r in range(len(SPLITS)) for sp in SPECIES}


def _ap_dict(probs, truth):
    top5 = [k for k, _ in sorted(probs.items(), key=lambda t: (-t[1], t[0]))[:5]]
    return ap_at_5(truth, top5)


def _eval_prob_map(pr_map):
    per_sp = {sp: [] for sp in SPECIES}
    overall = []
    for r in range(len(SPLITS)):
        ov = 0.0
        for sp in SPECIES:
            td = truths[(r, sp)]
            sc = float(np.mean([_ap_dict(pr_map[r][q], t) for q, t in td.items()]))
            per_sp[sp].append(sc)
            ov += W_TEST[sp] * sc
        overall.append(ov)
    return np.array(overall), {sp: np.array(per_sp[sp]) for sp in SPECIES}


ov_25, sp_25 = _eval_prob_map(cv_probs_v25)
ov_26, sp_26 = _eval_prob_map(cv_probs_v26)
print(f"\n[v25_hnsw_rescue] CV={ov_25.mean():.5f} ± {ov_25.std(ddof=1):.5f} (h={sp_25['hyena'].mean():.5f}, l={sp_25['leopard'].mean():.5f})")
print(f"[v26_dual_r2    ] CV={ov_26.mean():.5f} ± {ov_26.std(ddof=1):.5f} (h={sp_26['hyena'].mean():.5f}, l={sp_26['leopard'].mean():.5f})")

eq_map = {}
for r in range(len(SPLITS)):
    eq_map[r] = {}
    for sp in SPECIES:
        for q in truths[(r, sp)]:
            labs = set(cv_probs_v25[r][q]) | set(cv_probs_v26[r][q])
            eq_map[r][q] = {l: 0.5 * cv_probs_v25[r][q].get(l, 0.0) + 0.5 * cv_probs_v26[r][q].get(l, 0.0) for l in labs}
ov_eq, sp_eq = _eval_prob_map(eq_map)
best_single = ov_25 if ov_25.mean() >= ov_26.mean() else ov_26
d_eq = ov_eq - best_single
se_eq = d_eq.std(ddof=1) / np.sqrt(len(d_eq))
print(f"[Blend 0.50/0.50] CV={ov_eq.mean():.5f} ± {ov_eq.std(ddof=1):.5f} (h={sp_eq['hyena'].mean():.5f}, l={sp_eq['leopard'].mean():.5f}) | delta_vs_best={d_eq.mean():+.5f} (SE={se_eq:.5f})")

ws = np.round(np.arange(0.0, 1.01, 0.1), 2)
nested_sp, picked_w = {sp: [] for sp in SPECIES}, {sp: [] for sp in SPECIES}
for sp in SPECIES:
    mat = np.zeros((len(ws), len(SPLITS)))
    for wi, w in enumerate(ws):
        for r in range(len(SPLITS)):
            td = truths[(r, sp)]
            aps = []
            for q, t in td.items():
                labs = set(cv_probs_v25[r][q]) | set(cv_probs_v26[r][q])
                mix = {l: float(w) * cv_probs_v25[r][q].get(l, 0.0) + (1.0 - float(w)) * cv_probs_v26[r][q].get(l, 0.0) for l in labs}
                aps.append(_ap_dict(mix, t))
            mat[wi, r] = np.mean(aps)
    for r in range(len(SPLITS)):
        wi_b = int(np.argmax(np.delete(mat, r, axis=1).mean(1)))
        nested_sp[sp].append(float(mat[wi_b, r]))
        picked_w[sp].append(float(ws[wi_b]))

ov_nes = np.array([W_TEST["hyena"] * nested_sp["hyena"][r] + W_TEST["leopard"] * nested_sp["leopard"][r] for r in range(len(SPLITS))])
d_nes = ov_nes - best_single
se_nes = d_nes.std(ddof=1) / np.sqrt(len(d_nes))
print(f"[Nested Blend   ] CV={ov_nes.mean():.5f} ± {ov_nes.std(ddof=1):.5f} (h={np.mean(nested_sp['hyena']):.5f}, l={np.mean(nested_sp['leopard']):.5f}) | delta_vs_best={d_nes.mean():+.5f} (SE={se_nes:.5f}) | outer_w_v25={picked_w}")

# 3. Choose final blend weights and write validated submission.csv
if isinstance(BLEND_MODE, (int, float)):
    w_final = {sp: float(BLEND_MODE) for sp in SPECIES}
elif BLEND_MODE == "equal":
    w_final = {sp: 0.5 for sp in SPECIES}
else:
    w_final = {sp: float(np.median(picked_w[sp])) for sp in SPECIES}
    if ov_eq.mean() >= ov_nes.mean():
        w_final = {sp: 0.5 for sp in SPECIES}

allowed = DATA.train.groupby("species").identity.apply(set).to_dict()
sp_map = dict(zip(DATA.test.id, DATA.test.species))
final_preds, blend_probs = {}, {}
for q in DATA.test.id:
    sp = sp_map[q]
    wv = w_final[sp]
    labs = set(test_probs_v25[q]) | set(test_probs_v26[q])
    sc = {l: wv * test_probs_v25[q].get(l, 0.0) + (1.0 - wv) * test_probs_v26[q].get(l, 0.0) for l in labs}
    blend_probs[q] = sc
    ranked = [l for l in sorted(labs, key=lambda l: (-sc[l], l)) if l == NEW or l in allowed[sp]][:5]
    for l in [NEW] + sorted(allowed[sp]):
        if len(ranked) == 5:
            break
        if l not in ranked:
            ranked.append(l)
    final_preds[q] = ranked

sub_path = WORK_DIR / "submission.csv"
sub = validate_and_write_submission(final_preds, DATA.test, DATA.train, sub_path)
pickle.dump(blend_probs, open(WORK_DIR / "submission_blend.probs.pkl", "wb"))

first = sub.predictions.str.split(" ").str[0]
frac_new = {sp: round(float((first[DATA.test.species.values == sp] == NEW).mean()), 4) for sp in SPECIES}
print(f"\n[DONE] Wrote {sub_path} ({len(sub)} rows) | blend_w_v25={w_final} | new@rank1={frac_new} | total={time.time() - T_START:.0f}s")
sub.head(10)